# Sampling from the Classifier-Guided Diffusion Model

Version 1.0

Source:

https://github.com/openai/guided-diffusion/

Adapted by:

Antonio Esteves @ UMinho, April 2025
---
TODO:

* Modify `'OUR_WANDB_PROJECT_ID'`
* Modify `'OUR_WANDB_ENTITY'`
* In `get_config` function, modify `experiment.experiment_name`, `experiment.root_dir`, `data.data_path`, `data.dataset`,   `sampling.model_file`, `sampling.classifier_file`, `sampling.device`,   `evaluate.model_file`, `evaluate.data_path`, `evaluate.device`,   `classifier.data_path`, `classifier.val_path`.
---

In [ ]:
import ml_collections
import os
import math
import enum
import random
import copy
import time
import wandb

import numpy               as     np
from   abc                 import ABC, abstractmethod
from   functools           import partial
from   PIL                 import Image
from   collections         import defaultdict
import blobfile            as     bf
from   tqdm.notebook       import tqdm
from   tqdm                import trange

import torch
import torch.nn            as     nn
import torch.nn.functional as     F
from   torch._utils        import _flatten_dense_tensors, _unflatten_dense_tensors
from   torch.utils.data    import DataLoader, Dataset
from   torch.optim         import AdamW


## Configuration

In [ ]:
def get_config():
    '''
    Defines the configuration for training with LSUN dataset and Guided Diffusion model.
    '''
    config = ml_collections.ConfigDict()

    config.training   = training   = ml_collections.ConfigDict()
    config.data       = data       = ml_collections.ConfigDict()
    config.model      = model      = ml_collections.ConfigDict()
    config.optim      = optim      = ml_collections.ConfigDict()
    config.experiment = experiment = ml_collections.ConfigDict()
    config.sampling   = sampling   = ml_collections.ConfigDict()
    config.evaluate   = evaluate   = ml_collections.ConfigDict()
    config.classifier = classifier  = ml_collections.ConfigDict()

    experiment.experiment_name      = "GuidedDiffusion_04_sampling_ddpm1000_scale8_01"
    experiment.root_dir             = "OUR_WORK_DIR_HERE" # Root directory for the experiment
    experiment.results_dir          = "results"
    experiment.models_dir           = "models"
    experiment.mode                 = "train"   # "train" or "sample" or "bpd" or "summary"
    experiment.saved_model          = None      # File (without path) with a saved model, contained 
                                                # in 'models' folder, to be restored or 
                                                # 'None' to start from scratch.

    # training .....................................................

    training.batch_size                 = 120     # Batch size
    training.microbatch                 = 10      # -1 disables microbatches
    training.epochs                     = 50      # Number of epochs
    training.n_iters                    = None    # Number of iterations.
                                                  # It is calculated based on the number of epochs and dataset size.
    training.log_interval               = 100     # Interval between successive logs of training progress.
    training.snapshot_freq              = 1       # Interval between saving successive model checkpoints (in epochs). 
                                                  # Later it is updated to iterations.
    training.learn_sigma                = True
    training.diffusion_steps            = 1000
    training.noise_schedule             = "cosine"
    training.timestep_respacing         = [1000] # [1000], [500], [250], 'ddim100', 'ddim50', 'ddim25'
    training.use_kl                     = False
    training.predict_xstart             = False
    training.rescale_timesteps          = False
    training.rescale_learned_sigmas     = False
    training.use_fp16                   = False
    training.fp16_scale_growth          = 1e-3
    training.use_gradient_checkpointing = False

    # optimization ...................................................

    optim.weight_decay              = 0.0     # Weight decay.
    optim.optimizer                 = "Adam"  # The optimizer.
    optim.lr                        = 0.00005 # Learning rate.
    optim.lr_anneal_steps           = 0
    optim.beta1                     = 0.9     # Beta1 parameter of the Adam optimizer.

    # model ..........................................................

    model.schedule_sampler          = "uniform"
    model.ema_rate                  = "0.9995"  # comma-separated list of EMA values
    model.num_channels              = 128
    model.num_res_blocks            = 3
    model.num_heads                 = 8
    model.num_heads_upsample        = -1
    model.num_head_channels         = -1
    model.attention_resolutions     = "16,8"
    model.channel_mult              = ""
    model.dropout                   = 0.25
    model.class_cond                = True
    model.use_scale_shift_norm      = True
    model.resblock_updown           = False
    model.use_new_attention_order   = False

    # data ..........................................................

    data.dataset                     = "LSUN 4 classes"               # Dataset name.
    data.data_path                   = "OUR_DATASETS_ROOT/lsun128/train" # Path to the training dataset.
    data.image_size                  = 128                            # Image size.
    data.num_classes                 = 4                              # Number of classes.

    # sampling ......................................................
    sampling.clip_denoised           = True
    sampling.num_samples             = 40
    sampling.batch_size              = 4
    sampling.use_ddim                = False
    sampling.model_file              = 'OUR_WORK_DIR/models/GuidedDiffusion_04/GuidedDiffusion_04_epochXYZ_stepABC.pth' # File with a saved diffusion model.
    sampling.classifier_file         = 'OUR_WORK_DIR/models/GuidedDiffusion_classifier_01/GuidedDiffusion_classifier_01_epochXYZ_stepABC.pth'  # File with a saved classifier model.
    sampling.device                  = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')

    # evaluate .................... ..................................
    evaluate.clip_denoised           = True
    evaluate.num_samples             = 1000
    evaluate.batch_size              = 1
    evaluate.model_file              = None # File (without path) with a saved model, 
                                            # contained in 'models' folder, to be restored.
    evaluate.data_path               = "OUR_DATASETS_ROOT/lsun128/val"
    evaluate.device                  = 'cuda:0'

    classifier.data_path             = "OUR_DATASETS_ROOT/lsun128/train"
    classifier.val_path              = "OUR_DATASETS_ROOT/lsun128/val"  # None if not using validation
    classifier.use_fp16              = False
    classifier.width                 = 128
    classifier.depth                 = 2
    classifier.scale                 = 1.0        # 1.0, 4.0, 8.0
    classifier.attention_resolutions = "32,16,8"
    classifier.use_scale_shift_norm  = True
    classifier.resblock_updown       = True
    classifier.pool                  = "attention"
    classifier.noised                = True
    classifier.epochs                = 100
    classifier.iterations            = 0      # Calculated later
    classifier.lr                    = 0.0003
    classifier.weight_decay          = 0.0
    classifier.anneal_lr             = False
    classifier.batch_size            = 32
    classifier.microbatch            = -1
    classifier.schedule_sampler      = "uniform"
    classifier.resume_checkpoint     = ""
    classifier.log_interval          = 100  # in iterations
    classifier.eval_interval         = 5000 # in iterations
    classifier.save_interval         = 1    # in epochs

    config.device = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')

    return config

def print_config(config):
  '''
  Prints the configuration.
  '''
  print('Configuration parameters:')
  for name, values in config.items():
      if isinstance(values, ml_collections.config_dict.config_dict.ConfigDict):
          print(f'{name}:')
          for key, value in values.items():
              print(f'\t{key}: {value}')
      else:
          print(f'{name}: {values}')

## Setup the environment

Read the configuration and create the necessary folders.

In [ ]:
config = get_config()

print(f'Using {config.device} for computing')

# Print the configuration .................................................

print_config(config)

# Location where we will save the sampling results
RESULTS_PATH = os.path.join(
    config.experiment.root_dir,
    config.experiment.results_dir,
    config.experiment.experiment_name
)
os.makedirs(RESULTS_PATH, exist_ok=True)

# Location of the trained models
MODELS_PATH  = os.path.join(
    config.experiment.root_dir,
    config.experiment.models_dir,
    config.experiment.experiment_name
)

## Utility functions

In [ ]:
def time_format(seconds: int) -> str:
    '''
    Converts a time in seconds to days:hours:minutes:seconds.
    '''
    if seconds is not None:
        seconds = int(seconds)
        d = seconds // (3600 * 24)
        h = seconds // 3600 % 24
        m = seconds % 3600 // 60
        s = seconds % 3600 % 60
        if d > 0:
            return '{:02d}D {:02d}H {:02d}m {:02d}s'.format(d, h, m, s)
        elif h > 0:
            return '{:02d}H {:02d}m {:02d}s'.format(h, m, s)
        elif m > 0:
            return '{:02d}m {:02d}s'.format(m, s)
        elif s > 0:
            return '{:02d}s'.format(s)
    return '-'


## Various Utilities to Describe Neural Networks

In [ ]:
class GroupNorm32(nn.GroupNorm):
    def forward(self, x):
        return super().forward(x.float()).type(x.dtype)


def conv_nd(dims, *args, **kwargs):
    """
    Creates a 1D, 2D, or 3D convolution layer.
    """
    if dims == 1:
        return nn.Conv1d(*args, **kwargs)
    elif dims == 2:
        return nn.Conv2d(*args, **kwargs)
    elif dims == 3:
        return nn.Conv3d(*args, **kwargs)
    raise ValueError(f"unsupported dimensions: {dims}")


def linear(*args, **kwargs):
    """
    Creates a fully-connected layer.
    """
    return nn.Linear(*args, **kwargs)


def avg_pool_nd(dims, *args, **kwargs):
    """
    Creates a 1D, 2D, or 3D average pooling layer.
    """
    if dims == 1:
        return nn.AvgPool1d(*args, **kwargs)
    elif dims == 2:
        return nn.AvgPool2d(*args, **kwargs)
    elif dims == 3:
        return nn.AvgPool3d(*args, **kwargs)
    raise ValueError(f"unsupported dimensions: {dims}")


def update_ema(target_params, source_params, rate=0.99):
    """
    Updates target parameters to be closer to those of source parameters using
    an exponential moving average.

    Arguments:
    *  target_params: the target parameter sequence.
    *  source_params: the source parameter sequence.
    * rate:          the EMA rate (closer to 1 means slower).
    """
    for targ, src in zip(target_params, source_params):
        targ.detach().mul_(rate).add_(src, alpha=1 - rate)


def zero_module(module):
    """
    Zero out the parameters of a module and return it.
    """
    for p in module.parameters():
        p.detach().zero_()
    return module


def scale_module(module, scale):
    """
    Scales the parameters of a module and return it.
    """
    for p in module.parameters():
        p.detach().mul_(scale)
    return module


def mean_flat(tensor):
    """
    Takes the mean over all non-batch dimensions.
    """
    return tensor.mean(dim=list(range(1, len(tensor.shape))))


def normalization(channels):
    """
    Makes a standard normalization layer.

    Arguments:
    *  channels: Number of input channels.
    * return:         An nn.Module for normalization.
    """
    return GroupNorm32(32, channels)


def timestep_embedding(timesteps, dim, max_period=10000):
    """
    Creates a sinusoidal timestep embedding.

    Arguments:
    *  timesteps:  A 1-D Tensor of N indices, one per batch element.
                   These may be fractional.
    *  dim:        The dimension of the output.
    *  max_period: Controls the minimum frequency of the embeddings.

    Returns:
       An [N x dim] tensor of positional embeddings.
    """
    half  = dim // 2
    freqs = torch.exp(
        -math.log(max_period) * torch.arange(start=0, end=half, dtype=torch.float32) / half
    ).to(device=timesteps.device)
    args      = timesteps[:, None].float() * freqs[None]
    embedding = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
    if dim % 2:
        embedding = torch.cat([embedding, torch.zeros_like(embedding[:, :1])], dim=-1)

    return embedding


def checkpoint(func, inputs, params, flag):
    """
    Evaluates a function without caching intermediate activations, allowing for
    reduced memory at the expense of extra compute in the backward pass.

    Arguments:
    * func:   The function to evaluate.
    * inputs: The argument sequence to pass to `func`.
    * params: A sequence of parameters `func` depends on but does not
              explicitly take as arguments.
    * flag:   If False, disable gradient checkpointing.
    """
    if flag:
        args = tuple(inputs) + tuple(params)
        return CheckpointFunction.apply(func, len(inputs), *args)
    else:
        return func(*inputs)


class CheckpointFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, run_function, length, *args):
        ctx.run_function  = run_function
        ctx.input_tensors = list(args[:length])
        ctx.input_params  = list(args[length:])
        with torch.no_grad():
            output_tensors = ctx.run_function(*ctx.input_tensors)
        return output_tensors

    @staticmethod
    def backward(ctx, *output_grads):
        ctx.input_tensors = [x.detach().requires_grad_(True) for x in ctx.input_tensors]
        with torch.enable_grad():
            # Fixes a bug where the first op in run_function modifies the
            # Tensor storage in place, which is not allowed for detach()'d
            # Tensors.
            shallow_copies = [x.view_as(x) for x in ctx.input_tensors]
            output_tensors = ctx.run_function(*shallow_copies)
        input_grads = torch.autograd.grad(
            output_tensors,
            ctx.input_tensors + ctx.input_params,
            output_grads,
            allow_unused=True,
        )
        del ctx.input_tensors
        del ctx.input_params
        del output_tensors
        return (None, None) + input_grads


## Helpers to train with 16-bit precision

In [ ]:
INITIAL_LOG_LOSS_SCALE = 20.0

def convert_module_to_f16(l):
    """
    Converts weights and biases of the convolution layer 'l' to float16.
    """
    if isinstance(l, (nn.Conv1d, nn.Conv2d, nn.Conv3d)):
        l.weight.data = l.weight.data.half()
        if l.bias is not None:
            l.bias.data = l.bias.data.half()


def convert_module_to_f32(l):
    """
    Converts weights and biases of the convolution layer 'l' back to float32, 
    undoing what convert_module_to_f16() did.
    """
    if isinstance(l, (nn.Conv1d, nn.Conv2d, nn.Conv3d)):
        l.weight.data = l.weight.data.float()
        if l.bias is not None:
            l.bias.data = l.bias.data.float()


## Samplers

In [ ]:
def create_named_schedule_sampler(name, diffusion):
    """
    Create a ScheduleSampler from a library of pre-defined samplers.

    Arguments:
    * name:      the name of the sampler.
    * diffusion: the diffusion object to sample for.
    """
    if name == "uniform":
        return UniformSampler(diffusion)
    elif name == "loss-second-moment":
        return LossSecondMomentResampler(diffusion)
    else:
        raise NotImplementedError(f"[WARN] unknown schedule sampler {name}")


class ScheduleSampler(ABC):
    """
    A distribution over timesteps in the diffusion process, intended to reduce
    variance of the objective.

    By default, samplers perform unbiased importance sampling, in which the
    objective's mean is unchanged.
    However, subclasses may override sample() to change how the resampled
    terms are reweighted, allowing for actual changes in the objective.
    """

    @abstractmethod
    def weights(self):
        """
        Get a numpy array of weights, one per diffusion step.

        The weights do not have to be normalized, but must be positive.
        """

    def sample(self, batch_size, device):
        """
        Importance-sample timesteps for a batch.

        Arguments:
        * batch_size: the number of timesteps.
        * device:     the torch device to save to.
        Returns: a tuple (timesteps, weights):
                 - timesteps: a tensor of timestep indices.
                 - weights:   a tensor of weights to scale the resulting losses.
        """
        w          = self.weights()
        p          = w / np.sum(w)
        indices_np = np.random.choice(len(p), size=(batch_size), p=p)
        indices    = torch.from_numpy(indices_np).long().to(device)
        weights_np = 1 / (len(p) * p[indices_np])
        weights    = torch.from_numpy(weights_np).float().to(device)
        return indices, weights


class UniformSampler(ScheduleSampler):
    def __init__(self, diffusion):
        self.diffusion = diffusion
        self._weights  = np.ones([diffusion.num_timesteps])

    def weights(self):
        return self._weights


class LossAwareSampler(ScheduleSampler):
    def update_with_local_losses(self, local_ts, local_losses):
        """
        Update the reweighting using losses from a model.

        Originally, this method was designed to work in  distributed training,
        and to perform synchronization to make sure all of the ranks
        maintain the exact same reweighting.

        Arguments:
        * local_ts:     an integer Tensor of timesteps.
        * local_losses: a 1D Tensor of losses.
        """
        batch_sizes = [
            torch.tensor([0], dtype=torch.int32, device=local_ts.device)
        ]
        max_bs      = batch_sizes

        timestep_batches = [torch.zeros(max_bs).to(local_ts)]
        loss_batches     = [torch.zeros(max_bs).to(local_losses)]

        b0        = batch_sizes[0]
        t_b_0     = timestep_batches[0][:b0].item()
        timesteps = [t_b_0]
        l_b_0     = loss_batches[0][:b0].item()
        losses    = [l_b_0]

        self.update_with_all_losses(timesteps, losses)

    @abstractmethod
    def update_with_all_losses(self, ts, losses):
        """
        Update the reweighting using losses from a model.

        Sub-classes should override this method to update the reweighting
        using losses from the model.

        This method directly updates the reweighting without synchronizing
        between workers. It is called by update_with_local_losses from all
        ranks with identical arguments. Thus, it should have deterministic
        behavior to maintain state across workers.

        * ts:     a list of int timesteps.
        * losses: a list of float losses, one per timestep.
        """


class LossSecondMomentResampler(LossAwareSampler):
    def __init__(self, diffusion, history_per_term=10, uniform_prob=0.001):
        self.diffusion        = diffusion
        self.history_per_term = history_per_term
        self.uniform_prob     = uniform_prob
        self._loss_history    = np.zeros(
            [diffusion.num_timesteps, history_per_term], dtype=np.float64
        )
        self._loss_counts     = np.zeros([diffusion.num_timesteps], dtype=np.int)

    def weights(self):
        if not self._warmed_up():
            return np.ones([self.diffusion.num_timesteps], dtype=np.float64)
        weights  = np.sqrt(np.mean(self._loss_history ** 2, axis=-1))
        weights /= np.sum(weights)
        weights *= 1 - self.uniform_prob
        weights += self.uniform_prob / len(weights)
        return weights

    def update_with_all_losses(self, ts, losses):
        for t, loss in zip(ts, losses):
            if self._loss_counts[t] == self.history_per_term:
                # Shift out the oldest loss term
                self._loss_history[t, :-1] = self._loss_history[t, 1:]
                self._loss_history[t, -1]  = loss
            else:
                self._loss_history[t, self._loss_counts[t]] = loss
                self._loss_counts[t] += 1

    def _warmed_up(self):
        return (self._loss_counts == self.history_per_term).all()

## Attention classes

In [ ]:
def count_flops_attn(model, _x, y):
    """
    A counter for the `thop` package to count the operations in an
    attention operation.
    Meant to be used like:
        macs, params = thop.profile(
            model,
            inputs=(inputs, timestamps),
            custom_ops={QKVAttention: QKVAttention.count_flops},
        )
    """
    b, c, *spatial   = y[0].shape
    num_spatial      = int(np.prod(spatial))
    # We perform two matmuls with the same number of ops.
    # The first computes the weight matrix, the second computes
    # the combination of the value vectors.
    matmul_ops       = 2 * b * (num_spatial ** 2) * c
    model.total_ops += torch.DoubleTensor([matmul_ops])

class AttentionPool2d(nn.Module):
    """
    Adapted from CLIP: https://github.com/openai/CLIP/blob/main/clip/model.py
    """

    def __init__(
        self,
        spacial_dim:        int,
        embed_dim:          int,
        num_heads_channels: int,
        output_dim:         int = None,
    ):
        super().__init__()
        self.positional_embedding = nn.Parameter(
            torch.randn(embed_dim, spacial_dim ** 2 + 1) / embed_dim ** 0.5
        )
        self.qkv_proj  = conv_nd(1, embed_dim, 3 * embed_dim, 1)
        self.c_proj    = conv_nd(1, embed_dim, output_dim or embed_dim, 1)
        self.num_heads = embed_dim // num_heads_channels
        self.attention = QKVAttention(self.num_heads)

    def forward(self, x):
        b, c, *_spatial = x.shape
        x = x.reshape(b, c, -1)  # NC(HW)
        x = torch.cat([x.mean(dim=-1, keepdim=True), x], dim=-1)  # NC(HW+1)
        x = x + self.positional_embedding[None, :, :].to(x.dtype)  # NC(HW+1)
        x = self.qkv_proj(x)
        x = self.attention(x)
        x = self.c_proj(x)
        return x[:, :, 0]


class TimestepBlock(nn.Module):
    """
    Any module where forward() takes timestep embeddings as a second argument.
    """

    @abstractmethod
    def forward(self, x, emb):
        """
        Apply the module to `x` given `emb` timestep embeddings.
        """


class TimestepEmbedSequential(nn.Sequential, TimestepBlock):
    """
    A sequential module that passes timestep embeddings to the children that
    support it as an extra input.
    """

    def forward(self, x, emb):
        for layer in self:
            if isinstance(layer, TimestepBlock):
                x = layer(x, emb)
            else:
                x = layer(x)
        return x


class AttentionBlock(nn.Module):
    """
    An attention block that allows spatial positions to attend to each other.

    Originally ported from here, but adapted to the N-d case.
    https://github.com/hojonathanho/diffusion/blob/1e0dceb3b3495bbe19116a5e1b3596cd0706c543/diffusion_tf/models/unet.py#L66.
    """

    def __init__(
        self,
        channels,
        num_heads               = 1,
        num_head_channels       = -1,
        use_checkpoint          = False,
        use_new_attention_order = False,
    ):
        super().__init__()
        self.channels = channels
        if num_head_channels == -1:
            self.num_heads = num_heads
        else:
            assert (
                channels % num_head_channels == 0
            ), f"q,k,v channels {channels} is not divisible by num_head_channels {num_head_channels}"
            self.num_heads = channels // num_head_channels
        self.use_checkpoint = use_checkpoint
        self.norm           = normalization(channels)
        self.qkv            = conv_nd(1, channels, channels * 3, 1)
        if use_new_attention_order:
            # split qkv before split heads
            self.attention = QKVAttention(self.num_heads)
        else:
            # split heads before split qkv
            self.attention = QKVAttentionLegacy(self.num_heads)

        self.proj_out = zero_module(conv_nd(1, channels, channels, 1))

    def forward(self, x):
        return checkpoint(self._forward, (x,), self.parameters(), True)

    def _forward(self, x):
        b, c, *spatial = x.shape
        x              = x.reshape(b, c, -1)
        qkv            = self.qkv(self.norm(x))
        h              = self.attention(qkv)
        h              = self.proj_out(h)
        return (x + h).reshape(b, c, *spatial)


class QKVAttentionLegacy(nn.Module):
    """
    A module which performs QKV attention. Matches legacy QKVAttention + input/ouput heads shaping.
    """

    def __init__(self, n_heads):
        super().__init__()
        self.n_heads = n_heads

    def forward(self, qkv):
        """
        Apply QKV attention.

        Arguments:
        * qkv:   an [N x (H * 3 * C) x T] tensor of Qs, Ks, and Vs.
        Returns: an [N x (H * C) x T] tensor after attention.
        """
        bs, width, length = qkv.shape
        assert width % (3 * self.n_heads) == 0
        ch      = width // (3 * self.n_heads)
        q, k, v = qkv.reshape(bs * self.n_heads, ch * 3, length).split(ch, dim=1)
        scale   = 1 / math.sqrt(math.sqrt(ch))
        weight  = torch.einsum(
            "bct,bcs->bts", q * scale, k * scale
        )  # More stable with f16 than dividing afterwards
        weight = torch.softmax(weight.float(), dim=-1).type(weight.dtype)
        a = torch.einsum("bts,bcs->bct", weight, v)
        return a.reshape(bs, -1, length)

    @staticmethod
    def count_flops(model, _x, y):
        return count_flops_attn(model, _x, y)


class QKVAttention(nn.Module):
    """
    A module which performs QKV attention and splits in a different order.
    """

    def __init__(self, n_heads):
        super().__init__()
        self.n_heads = n_heads

    def forward(self, qkv):
        """
        Apply QKV attention.

        Arguments:
        * qkv:   an [N x (3 * H * C) x T] tensor of Qs, Ks, and Vs.
        Returns: an [N x (H * C) x T] tensor after attention.
        """
        bs, width, length = qkv.shape
        assert width % (3 * self.n_heads) == 0
        ch      = width // (3 * self.n_heads)
        q, k, v = qkv.chunk(3, dim=1)
        scale   = 1 / math.sqrt(math.sqrt(ch))
        weight  = torch.einsum(
            "bct,bcs->bts",
            (q * scale).view(bs * self.n_heads, ch, length),
            (k * scale).view(bs * self.n_heads, ch, length),
        )  # More stable with f16 than dividing afterwards
        weight  = torch.softmax(weight.float(), dim=-1).type(weight.dtype)
        a       = torch.einsum("bts,bcs->bct", weight, v.reshape(bs * self.n_heads, ch, length))
        return a.reshape(bs, -1, length)

    @staticmethod
    def count_flops(model, _x, y):
        return count_flops_attn(model, _x, y)


## Losses

In [ ]:
def normal_kl(mean1, logvar1, mean2, logvar2):
    """
    Compute the KL divergence between two Gaussian.

    Shapes are automatically broadcasted, so batches can be compared to
    scalars, among other use cases.
    """
    tensor = None
    for obj in (mean1, logvar1, mean2, logvar2):
        if isinstance(obj, torch.Tensor):
            tensor = obj
            break
    assert tensor is not None, "at least one argument must be a Tensor"

    # Force variances to be Tensors. Broadcasting helps convert scalars to
    # Tensors, but it does not work for torch.exp().
    logvar1, logvar2 = [
        x if isinstance(x, torch.Tensor) else torch.tensor(x).to(tensor)
        for x in (logvar1, logvar2)
    ]

    return 0.5 * (
        -1.0
        + logvar2
        - logvar1
        + torch.exp(logvar1 - logvar2)
        + ((mean1 - mean2) ** 2) * torch.exp(-logvar2)
    )


def approx_standard_normal_cdf(x):
    """
    A fast approximation of the cumulative distribution function of the
    standard normal.
    """
    return 0.5 * (1.0 + torch.tanh(np.sqrt(2.0 / np.pi) * (x + 0.044715 * torch.pow(x, 3))))


def discretized_gaussian_log_likelihood(x, *, means, log_scales):
    """
    Compute the log-likelihood of a Gaussian distribution discretizing to a
    given image.

    Arguments:
    * x:          the target images. It is assumed that this was uint8 values,
                  rescaled to the range [-1, 1].
    * means:      the Gaussian mean Tensor.
    * log_scales: the Gaussian log stddev Tensor.
    Returns:      a tensor like x of log probabilities (in nats).
    """
    assert x.shape == means.shape == log_scales.shape
    centered_x            = x - means
    inv_stdv              = torch.exp(-log_scales)
    plus_in               = inv_stdv * (centered_x + 1.0 / 255.0)
    cdf_plus              = approx_standard_normal_cdf(plus_in)
    min_in                = inv_stdv * (centered_x - 1.0 / 255.0)
    cdf_min               = approx_standard_normal_cdf(min_in)
    log_cdf_plus          = torch.log(cdf_plus.clamp(min=1e-12))
    log_one_minus_cdf_min = torch.log((1.0 - cdf_min).clamp(min=1e-12))
    cdf_delta             = cdf_plus - cdf_min
    log_probs             = torch.where(
        x < -0.999,
        log_cdf_plus,
        torch.where(x > 0.999, log_one_minus_cdf_min, torch.log(cdf_delta.clamp(min=1e-12))),
    )
    assert log_probs.shape == x.shape
    return log_probs


## Gaussian Diffusion

In [ ]:
def betas_for_alpha_bar(num_diffusion_timesteps, alpha_bar, max_beta=0.999):
    """
    Create a beta schedule that discretizes the given alpha_t_bar function,
    which defines the cumulative product of (1-beta) over time from t = [0,1].

    * num_diffusion_timesteps: the number of betas to produce.
    * alpha_bar: a lambda that takes an argument t from 0 to 1 and
                 produces the cumulative product of (1-beta) up to that
                 part of the diffusion process.
    * max_beta:  the maximum beta to use; use values lower than 1 to
                 prevent singularities.
    """
    betas = []
    for i in range(num_diffusion_timesteps):
        t1 = i / num_diffusion_timesteps
        t2 = (i + 1) / num_diffusion_timesteps
        betas.append(min(1 - alpha_bar(t2) / alpha_bar(t1), max_beta))
    return np.array(betas)


def get_named_beta_schedule(schedule_name, num_diffusion_timesteps):
    """
    Get a pre-defined beta schedule given a schedule name.

    The beta schedule library consists of beta schedules which remain similar
    in the limit of num_diffusion_timesteps.
    Beta schedules may be added, but should not be removed or changed once
    they are committed to maintain backwards compatibility.
    """
    if schedule_name == "linear":
        # Linear schedule from Ho et al, extended to work for any number of
        # diffusion steps.
        scale = 1000 / num_diffusion_timesteps
        beta_start = scale * 0.0001
        beta_end = scale * 0.02
        return np.linspace(
            beta_start, beta_end, num_diffusion_timesteps, dtype=np.float64
        )
    elif schedule_name == "cosine":
        return betas_for_alpha_bar(
            num_diffusion_timesteps,
            lambda t: math.cos((t + 0.008) / 1.008 * math.pi / 2) ** 2,
        )
    else:
        raise NotImplementedError(f"unknown beta schedule: {schedule_name}")


def _extract_into_tensor(arr, timesteps, broadcast_shape):
    """
    Extract values from a 1-D numpy array for a batch of indices.

    Arguments:
    * arr:             the 1-D numpy array.
    * timesteps:       a tensor of indices into the array to extract.
    * broadcast_shape: a larger shape of K dimensions with the batch
                       dimension equal to the length of timesteps.
    Returns:           a tensor of shape [batch_size, 1, ...] where the shape has K dims.
    """
    res = torch.from_numpy(arr).to(device=timesteps.device)[timesteps].float()
    while len(res.shape) < len(broadcast_shape):
        res = res[..., None]
    return res.expand(broadcast_shape)

In [ ]:
class ModelMeanType(enum.Enum):
    """
    Which type of output the model predicts.
    """

    PREVIOUS_X = enum.auto()  # the model predicts x_{t-1}
    START_X    = enum.auto()  # the model predicts x_0
    EPSILON    = enum.auto()  # the model predicts epsilon


class ModelVarType(enum.Enum):
    """
    What is used as the model's output variance.

    The LEARNED_RANGE option has been added to allow the model to predict
    values between FIXED_SMALL and FIXED_LARGE, making its job easier.
    """

    LEARNED       = enum.auto()
    FIXED_SMALL   = enum.auto()
    FIXED_LARGE   = enum.auto()
    LEARNED_RANGE = enum.auto()


class LossType(enum.Enum):
    MSE = enum.auto()  # use raw MSE loss (and KL when learning variances)
    RESCALED_MSE = (
        enum.auto()
    )  # use raw MSE loss (with RESCALED_KL when learning variances)
    KL = enum.auto()  # use the variational lower-bound
    RESCALED_KL = enum.auto()  # like KL, but rescale to estimate the full VLB

    def is_vb(self):
        return self == LossType.KL or self == LossType.RESCALED_KL


In [ ]:
class GaussianDiffusion:
    """
    Utilities for training and sampling a diffusion model.

    Ported directly from here, and then adapted over time to further experimentation.
    https://github.com/hojonathanho/diffusion/blob/1e0dceb3b3495bbe19116a5e1b3596cd0706c543/diffusion_tf/diffusion_utils_2.py#L42

    Attributes:
    * betas:             a 1-D numpy array of betas for each diffusion timestep,
                         starting at T and going to 1.
    * model_mean_type:   a ModelMeanType determining what the model outputs.
    * model_var_type:    a ModelVarType determining how variance is output.
    * loss_type:         a LossType determining the loss function to use.
    * rescale_timesteps: if True, pass floating point timesteps into the
                         model so that they are always scaled like in the
                         original paper (0 to 1000).
    """

    def __init__(
            self,
            *,
            betas,
            model_mean_type,
            model_var_type,
            loss_type,
            rescale_timesteps = False,
        ):
        self.model_mean_type   = model_mean_type
        self.model_var_type    = model_var_type
        self.loss_type         = loss_type
        self.rescale_timesteps = rescale_timesteps

        # Use float64 for accuracy
        betas      = np.array(betas, dtype=np.float64)
        self.betas = betas
        assert len(betas.shape) == 1, "betas must be 1-D"
        assert (betas > 0).all() and (betas <= 1).all()

        self.num_timesteps = int(betas.shape[0])

        alphas                   = 1.0 - betas
        self.alphas_cumprod      = np.cumprod(alphas, axis=0)
        self.alphas_cumprod_prev = np.append(1.0, self.alphas_cumprod[:-1])
        self.alphas_cumprod_next = np.append(self.alphas_cumprod[1:], 0.0)
        assert self.alphas_cumprod_prev.shape == (self.num_timesteps,)

        # calculations for forward diffusion transition probability q(x_t | x_{t-1}) and others
        self.sqrt_alphas_cumprod           = np.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = np.sqrt(1.0 - self.alphas_cumprod)
        self.log_one_minus_alphas_cumprod  = np.log(1.0 - self.alphas_cumprod)
        self.sqrt_recip_alphas_cumprod     = np.sqrt(1.0 / self.alphas_cumprod)
        self.sqrt_recipm1_alphas_cumprod   = np.sqrt(1.0 / self.alphas_cumprod - 1)

        # calculations for reverse diffusion transition probability (posterior) q(x_{t-1} | x_t, x_0)
        self.posterior_variance = (
            betas * (1.0 - self.alphas_cumprod_prev) / (1.0 - self.alphas_cumprod)
        )
        # log calculation clipped because the posterior variance is 0 at the
        # beginning of the diffusion chain.
        self.posterior_log_variance_clipped = np.log(
            np.append(self.posterior_variance[1], self.posterior_variance[1:])
        )
        self.posterior_mean_coef1 = (
            betas * np.sqrt(self.alphas_cumprod_prev) / (1.0 - self.alphas_cumprod)
        )
        self.posterior_mean_coef2 = (
            (1.0 - self.alphas_cumprod_prev)
            * np.sqrt(alphas)
            / (1.0 - self.alphas_cumprod)
        )

    def q_mean_variance(self, x_start, t):
        """
        Get the distribution q(x_t | x_0).

        Arguments:
        * x_start: the [N x C x ...] tensor of noiseless inputs.
        * t:       the number of diffusion steps (minus 1). Here, 0 means one step.
        Returns:   A tuple (mean, variance, log_variance), all of x_start's shape.
        """
        mean = (
            _extract_into_tensor(self.sqrt_alphas_cumprod, t, x_start.shape) * x_start
        )
        variance     = _extract_into_tensor(1.0 - self.alphas_cumprod, t, x_start.shape)
        log_variance = _extract_into_tensor(
            self.log_one_minus_alphas_cumprod, t, x_start.shape
        )
        return mean, variance, log_variance

    def q_sample(self, x_start, t, noise=None):
        """
        Diffuse the data for a given number of diffusion steps.

        In other words, sample from q(x_t | x_0).

        Arguments:
        * x_start: the initial data batch.
        * t:       the number of diffusion steps (minus 1). Here, 0 means one step.
        * noise:   if specified, the split-out normal noise.
        Returns:   A noisy version of x_start.
        """
        if noise is None:
            noise = torch.randn_like(x_start)
        assert noise.shape == x_start.shape
        return (
            _extract_into_tensor(self.sqrt_alphas_cumprod, t, x_start.shape) * x_start
            + _extract_into_tensor(self.sqrt_one_minus_alphas_cumprod, t, x_start.shape)
            * noise
        )

    def q_posterior_mean_variance(self, x_start, x_t, t):
        """
        Compute the mean and variance of the diffusion posterior:

            q(x_{t-1} | x_t, x_0)

        """
        assert x_start.shape == x_t.shape
        posterior_mean = (
            _extract_into_tensor(self.posterior_mean_coef1, t, x_t.shape) * x_start
            + _extract_into_tensor(self.posterior_mean_coef2, t, x_t.shape) * x_t
        )
        posterior_variance = _extract_into_tensor(self.posterior_variance, t, x_t.shape)
        posterior_log_variance_clipped = _extract_into_tensor(
            self.posterior_log_variance_clipped, t, x_t.shape
        )
        assert (
            posterior_mean.shape[0]
            == posterior_variance.shape[0]
            == posterior_log_variance_clipped.shape[0]
            == x_start.shape[0]
        )
        return posterior_mean, posterior_variance, posterior_log_variance_clipped

    def p_mean_variance(
        self, model, x, t, clip_denoised=True, denoised_fn=None, model_kwargs=None
    ):
        """
        Apply the model to get p(x_{t-1} | x_t), as well as a prediction of
        the initial x, x_0.

        * model:         the model, which takes a signal and a batch of
                         timesteps as input.
        * x:             the [N x C x ...] tensor at time t.
        * t:             a 1-D Tensor of timesteps.
        * clip_denoised: if True, clip the denoised signal into [-1, 1].
        * denoised_fn:   if not None, a function which applies to the
                         x_start prediction before it is used to sample. 
                         Applies before clip_denoised.
        * model_kwargs:  if not None, a dict of extra keyword arguments to
                         pass to the model. This can be used for conditioning.
        Returns: a dict with the following keys:
                 - 'mean': the model mean output.
                 - 'variance': the model variance output.
                 - 'log_variance': the log of 'variance'.
                 - 'pred_xstart': the prediction for x_0.
        """
        if model_kwargs is None:
            model_kwargs = {}

        B, C = x.shape[:2]
        assert t.shape == (B,)
        model_output = model(x, self._scale_timesteps(t), **model_kwargs)

        if self.model_var_type in [ModelVarType.LEARNED, ModelVarType.LEARNED_RANGE]:
            assert model_output.shape == (B, C * 2, *x.shape[2:])
            model_output, model_var_values = torch.split(model_output, C, dim=1)
            if self.model_var_type == ModelVarType.LEARNED:
                model_log_variance = model_var_values
                model_variance = torch.exp(model_log_variance)
            else:
                min_log = _extract_into_tensor(
                    self.posterior_log_variance_clipped, t, x.shape
                )
                max_log = _extract_into_tensor(np.log(self.betas), t, x.shape)
                # The model_var_values is [-1, 1] for [min_var, max_var].
                frac = (model_var_values + 1) / 2
                model_log_variance = frac * max_log + (1 - frac) * min_log
                model_variance = torch.exp(model_log_variance)
        else:
            model_variance, model_log_variance = {
                # for fixedlarge, we set the initial (log-)variance like so
                # to get a better decoder log likelihood.
                ModelVarType.FIXED_LARGE: (
                    np.append(self.posterior_variance[1], self.betas[1:]),
                    np.log(np.append(self.posterior_variance[1], self.betas[1:])),
                ),
                ModelVarType.FIXED_SMALL: (
                    self.posterior_variance,
                    self.posterior_log_variance_clipped,
                ),
            }[self.model_var_type]
            model_variance = _extract_into_tensor(model_variance, t, x.shape)
            model_log_variance = _extract_into_tensor(model_log_variance, t, x.shape)

        def process_xstart(x):
            if denoised_fn is not None:
                x = denoised_fn(x)
            if clip_denoised:
                return x.clamp(-1, 1)
            return x

        if self.model_mean_type == ModelMeanType.PREVIOUS_X:
            pred_xstart = process_xstart(
                self._predict_xstart_from_xprev(x_t=x, t=t, xprev=model_output)
            )
            model_mean = model_output
        elif self.model_mean_type in [ModelMeanType.START_X, ModelMeanType.EPSILON]:
            if self.model_mean_type == ModelMeanType.START_X:
                pred_xstart = process_xstart(model_output)
            else:
                pred_xstart = process_xstart(
                    self._predict_xstart_from_eps(x_t=x, t=t, eps=model_output)
                )
            model_mean, _, _ = self.q_posterior_mean_variance(
                x_start=pred_xstart, x_t=x, t=t
            )
        else:
            raise NotImplementedError(self.model_mean_type)

        assert (
            model_mean.shape == model_log_variance.shape == pred_xstart.shape == x.shape
        )
        return {
            "mean": model_mean,
            "variance": model_variance,
            "log_variance": model_log_variance,
            "pred_xstart": pred_xstart,
        }

    def _predict_xstart_from_eps(self, x_t, t, eps):
        assert x_t.shape == eps.shape
        return (
            _extract_into_tensor(self.sqrt_recip_alphas_cumprod, t, x_t.shape) * x_t
            - _extract_into_tensor(self.sqrt_recipm1_alphas_cumprod, t, x_t.shape) * eps
        )

    def _predict_xstart_from_xprev(self, x_t, t, xprev):
        assert x_t.shape == xprev.shape
        return (  # (xprev - coef2*x_t) / coef1
            _extract_into_tensor(1.0 / self.posterior_mean_coef1, t, x_t.shape) * xprev
            - _extract_into_tensor(
                self.posterior_mean_coef2 / self.posterior_mean_coef1, t, x_t.shape
            )
            * x_t
        )

    def _predict_eps_from_xstart(self, x_t, t, pred_xstart):
        return (
            _extract_into_tensor(self.sqrt_recip_alphas_cumprod, t, x_t.shape) * x_t
            - pred_xstart
        ) / _extract_into_tensor(self.sqrt_recipm1_alphas_cumprod, t, x_t.shape)

    def _scale_timesteps(self, t):
        if self.rescale_timesteps:
            return t.float() * (1000.0 / self.num_timesteps)
        return t

    def condition_mean(self, cond_fn, p_mean_var, x, t, model_kwargs=None):
        """
        Compute the mean for the previous step, given a function cond_fn that
        computes the gradient of a conditional log probability with respect to
        x. In particular, cond_fn computes grad(log(p(y|x))), and we want to
        condition on y.

        This uses the conditioning strategy from Sohl-Dickstein et al. (2015).
        """

        gradient = cond_fn(x, self._scale_timesteps(t), **model_kwargs)
        new_mean = (
            p_mean_var["mean"].float() + p_mean_var["variance"] * gradient.float()
        )
        return new_mean

    def condition_score(self, cond_fn, p_mean_var, x, t, model_kwargs=None):
        """
        Compute what the p_mean_variance output would have been, should the
        model's score function be conditioned by cond_fn.

        See condition_mean() for details on cond_fn.

        Unlike condition_mean(), this instead uses the conditioning strategy
        from Song et al (2020).
        """
        alpha_bar = _extract_into_tensor(self.alphas_cumprod, t, x.shape)

        eps = self._predict_eps_from_xstart(x, t, p_mean_var["pred_xstart"])
        eps = eps - (1 - alpha_bar).sqrt() * cond_fn(
            x, self._scale_timesteps(t), **model_kwargs
        )

        out = p_mean_var.copy()
        out["pred_xstart"] = self._predict_xstart_from_eps(x, t, eps)
        out["mean"], _, _ = self.q_posterior_mean_variance(
            x_start=out["pred_xstart"], x_t=x, t=t
        )
        return out

    def p_sample(
            self,
            model,
            x,
            t,
            clip_denoised = True,
            denoised_fn   = None,
            cond_fn       = None,
            model_kwargs  = None,
        ):
        """
        Sample x_{t-1} from the model at the given timestep.

        Arguments:
        * model:         the model to sample from.
        * x:             the current tensor at x_{t-1}.
        * t:             the value of t, starting at 0 for the first diffusion step.
        * clip_denoised: if True, clip the x_start prediction to [-1, 1].
        * denoised_fn:   if not None, a function which applies to the
                         x_start prediction before it is used to sample.
        * cond_fn:       if not None, this is a gradient function that acts
                         similarly to the model.
        * model_kwargs:  if not None, a dict of extra keyword arguments to
                         pass to the model. This can be used for conditioning.
        Returns: a dict containing the following keys:
                 - 'sample': a random sample from the model.
                 - 'pred_xstart': a prediction of x_0.
        """
        out = self.p_mean_variance(
            model,
            x,
            t,
            clip_denoised = clip_denoised,
            denoised_fn   = denoised_fn,
            model_kwargs  = model_kwargs,
        )
        noise = torch.randn_like(x)
        nonzero_mask = (
            (t != 0).float().view(-1, *([1] * (len(x.shape) - 1)))
        )  # no noise when t == 0
        if cond_fn is not None:
            out["mean"] = self.condition_mean(
                cond_fn, out, x, t, model_kwargs=model_kwargs
            )
        sample = out["mean"] + nonzero_mask * torch.exp(0.5 * out["log_variance"]) * noise
        return {"sample": sample, "pred_xstart": out["pred_xstart"]}

    def p_sample_loop(
        self,
        model,
        shape,
        noise=None,
        clip_denoised=True,
        denoised_fn=None,
        cond_fn=None,
        model_kwargs=None,
        device=None,
        progress=False,
    ):
        """
        Generate samples from the model.

        Arguments:
        * model:         the model module.
        * shape:         the shape of the samples, (N, C, H, W).
        * noise:         if specified, the noise from the encoder to sample.
                         Should be of the same shape as `shape`.
        * clip_denoised: if True, clip x_start predictions to [-1, 1].
        * denoised_fn:   if not None, a function which applies to the
                         x_start prediction before it is used to sample.
        * cond_fn:       if not None, this is a gradient function that acts
                         similarly to the model.
        * model_kwargs:  if not None, a dict of extra keyword arguments to
                         pass to the model. This can be used for conditioning.
        * device:        if specified, the device to create the samples on.
                         If not specified, use a model parameter's device.
        * progress:      if True, show a tqdm progress bar.
        Returns:         a non-differentiable batch of samples.
        """
        final = None
        for sample in self.p_sample_loop_progressive(
            model,
            shape,
            noise=noise,
            clip_denoised = clip_denoised,
            denoised_fn   = denoised_fn,
            cond_fn       = cond_fn,
            model_kwargs  = model_kwargs,
            device        = device,
            progress      = progress,
        ):
            final = sample
        return final["sample"]

    def p_sample_loop_progressive(
        self,
        model,
        shape,
        noise         = None,
        clip_denoised = True,
        denoised_fn   = None,
        cond_fn       = None,
        model_kwargs  = None,
        device        = None,
        progress      = False,
    ):
        """
        Generate samples from the model and yield intermediate samples from
        each timestep of diffusion.

        Arguments are the same as p_sample_loop().
        Returns a generator over dicts, where each dict is the return value of
        p_sample().
        """
        if device is None:
            device = next(model.parameters()).device
        assert isinstance(shape, (tuple, list))
        if noise is not None:
            img = noise
        else:
            img = torch.randn(*shape, device=device)
        indices = list(range(self.num_timesteps))[::-1]

        if progress:
            # Lazy import so that we do not depend on tqdm.
            from tqdm.auto import tqdm

            indices = tqdm(indices)

        for i in indices:
            t = torch.tensor([i] * shape[0], device=device)

            with torch.no_grad():
                out = self.p_sample(
                    model,
                    img,
                    t,
                    clip_denoised=clip_denoised,
                    denoised_fn=denoised_fn,
                    cond_fn=cond_fn,
                    model_kwargs=model_kwargs,
                )
                yield out
                img = out["sample"]

    def ddim_sample(
        self,
        model,
        x,
        t,
        clip_denoised = True,
        denoised_fn   = None,
        cond_fn       = None,
        model_kwargs  = None,
        eta           = 0.0,
    ):
        """
        Sample x_{t-1} from the model using DDIM.

        Same usage as p_sample().
        """
        out = self.p_mean_variance(
            model,
            x,
            t,
            clip_denoised = clip_denoised,
            denoised_fn   = denoised_fn,
            model_kwargs  = model_kwargs,
        )
        if cond_fn is not None:
            out = self.condition_score(cond_fn, out, x, t, model_kwargs=model_kwargs)

        # Usually our model outputs epsilon, but we re-derive it
        # in case we used x_start or x_prev prediction.
        eps            = self._predict_eps_from_xstart(x, t, out["pred_xstart"])

        alpha_bar      = _extract_into_tensor(self.alphas_cumprod, t, x.shape)
        alpha_bar_prev = _extract_into_tensor(self.alphas_cumprod_prev, t, x.shape)
        sigma = (
            eta
            * torch.sqrt((1 - alpha_bar_prev) / (1 - alpha_bar))
            * torch.sqrt(1 - alpha_bar / alpha_bar_prev)
        )
        # Equation 12.
        noise     = torch.randn_like(x)
        mean_pred = (
            out["pred_xstart"] * torch.sqrt(alpha_bar_prev)
            + torch.sqrt(1 - alpha_bar_prev - sigma ** 2) * eps
        )
        nonzero_mask = (
            (t != 0).float().view(-1, *([1] * (len(x.shape) - 1)))
        )  # no noise when t == 0
        sample = mean_pred + nonzero_mask * sigma * noise
        return {"sample": sample, "pred_xstart": out["pred_xstart"]}

    def ddim_reverse_sample(
        self,
        model,
        x,
        t,
        clip_denoised = True,
        denoised_fn   = None,
        model_kwargs  = None,
        eta           = 0.0,
    ):
        """
        Sample x_{t+1} from the model using DDIM reverse ODE.
        """
        assert eta == 0.0, "Reverse ODE only for deterministic path"
        out = self.p_mean_variance(
            model,
            x,
            t,
            clip_denoised = clip_denoised,
            denoised_fn   = denoised_fn,
            model_kwargs  = model_kwargs,
        )
        # Usually our model outputs epsilon, but we re-derive it
        # in case we used x_start or x_prev prediction.
        eps = (
            _extract_into_tensor(self.sqrt_recip_alphas_cumprod, t, x.shape) * x
            - out["pred_xstart"]
        ) / _extract_into_tensor(self.sqrt_recipm1_alphas_cumprod, t, x.shape)
        alpha_bar_next = _extract_into_tensor(self.alphas_cumprod_next, t, x.shape)

        # Equation 12. reversed
        mean_pred = (
            out["pred_xstart"] * torch.sqrt(alpha_bar_next)
            + torch.sqrt(1 - alpha_bar_next) * eps
        )

        return {"sample": mean_pred, "pred_xstart": out["pred_xstart"]}

    def ddim_sample_loop(
        self,
        model,
        shape,
        noise         = None,
        clip_denoised = True,
        denoised_fn   = None,
        cond_fn       = None,
        model_kwargs  = None,
        device        = None,
        progress      = False,
        eta           = 0.0,
    ):
        """
        Generate samples from the model using DDIM.

        Same usage as p_sample_loop().
        """
        final = None
        for sample in self.ddim_sample_loop_progressive(
            model,
            shape,
            noise=noise,
            clip_denoised=clip_denoised,
            denoised_fn=denoised_fn,
            cond_fn=cond_fn,
            model_kwargs=model_kwargs,
            device=device,
            progress=progress,
            eta=eta,
        ):
            final = sample
        return final["sample"]

    def ddim_sample_loop_progressive(
        self,
        model,
        shape,
        noise=None,
        clip_denoised=True,
        denoised_fn=None,
        cond_fn=None,
        model_kwargs=None,
        device=None,
        progress=False,
        eta=0.0,
    ):
        """
        Use DDIM to sample from the model and yield intermediate samples from
        each timestep of DDIM.

        Same usage as p_sample_loop_progressive().
        """
        if device is None:
            device = next(model.parameters()).device
        assert isinstance(shape, (tuple, list))
        if noise is not None:
            img = noise
        else:
            img = torch.randn(*shape, device=device)
        indices = list(range(self.num_timesteps))[::-1]

        if progress:
            # Lazy import so that we don't depend on tqdm.
            from tqdm.auto import tqdm

            indices = tqdm(indices)

        for i in indices:
            t = torch.tensor([i] * shape[0], device=device)
            with torch.no_grad():
                out = self.ddim_sample(
                    model,
                    img,
                    t,
                    clip_denoised=clip_denoised,
                    denoised_fn=denoised_fn,
                    cond_fn=cond_fn,
                    model_kwargs=model_kwargs,
                    eta=eta,
                )
                yield out
                img = out["sample"]

    def _vb_terms_bpd(
        self, model, x_start, x_t, t, clip_denoised=True, model_kwargs=None
    ):
        """
        Get a term for the variational lower-bound.

        The resulting units are bits (rather than nats, as one might expect).
        This allows for comparison to other papers.

        Returns: a dict with the following keys:
                 - 'output': a shape [N] tensor of NLLs or KLs.
                 - 'pred_xstart': the x_0 predictions.
        """
        true_mean, _, true_log_variance_clipped = self.q_posterior_mean_variance(
            x_start=x_start, x_t=x_t, t=t
        )
        out = self.p_mean_variance(
            model, x_t, t, clip_denoised=clip_denoised, model_kwargs=model_kwargs
        )
        kl = normal_kl(
            true_mean, true_log_variance_clipped, out["mean"], out["log_variance"]
        )
        kl = mean_flat(kl) / np.log(2.0)

        decoder_nll = -discretized_gaussian_log_likelihood(
            x_start, means=out["mean"], log_scales=0.5 * out["log_variance"]
        )
        assert decoder_nll.shape == x_start.shape
        decoder_nll = mean_flat(decoder_nll) / np.log(2.0)

        # At the first timestep return the decoder NLL,
        # otherwise return KL(q(x_{t-1}|x_t,x_0) || p(x_{t-1}|x_t))
        output = torch.where((t == 0), decoder_nll, kl)
        return {"output": output, "pred_xstart": out["pred_xstart"]}

    def training_losses(self, model, x_start, t, model_kwargs=None, noise=None):
        """
        Compute training losses for a single timestep.

        Arguments:
        * model:        the model to evaluate loss on.
        * x_start:      the [N x C x ...] tensor of inputs.
        * t:            a batch of timestep indices.
        * model_kwargs: if not None, a dict of extra keyword arguments to
                        pass to the model. This can be used for conditioning.
        * noise:        if specified, the specific Gaussian noise to try to remove.
        Returns:        a dict with the key "loss" containing a tensor of shape [N].
                        Some mean or variance settings may also have other keys.
        """
        if model_kwargs is None:
            model_kwargs = {}
        if noise is None:
            noise = torch.randn_like(x_start)
        x_t = self.q_sample(x_start, t, noise=noise)

        terms = {}

        if self.loss_type == LossType.KL or self.loss_type == LossType.RESCALED_KL:
            terms["loss"] = self._vb_terms_bpd(
                model         = model,
                x_start       = x_start,
                x_t           = x_t,
                t             = t,
                clip_denoised = False,
                model_kwargs  = model_kwargs,
            )["output"]
            if self.loss_type == LossType.RESCALED_KL:
                terms["loss"] *= self.num_timesteps

        elif self.loss_type == LossType.MSE or self.loss_type == LossType.RESCALED_MSE:
            model_output = model(x_t, self._scale_timesteps(t), **model_kwargs)

            if self.model_var_type in [
                ModelVarType.LEARNED,
                ModelVarType.LEARNED_RANGE,
                ]:
                B, C = x_t.shape[:2]
                assert model_output.shape == (B, C * 2, *x_t.shape[2:])
                model_output, model_var_values = torch.split(model_output, C, dim=1)
                # Learn the variance using the variational bound, but do not let
                # it affect our mean prediction.
                frozen_out = torch.cat([model_output.detach(), model_var_values], dim=1)
                terms["vb"] = self._vb_terms_bpd(
                    model=lambda *args, r=frozen_out: r,
                    x_start=x_start,
                    x_t=x_t,
                    t=t,
                    clip_denoised=False,
                )["output"]
                if self.loss_type == LossType.RESCALED_MSE:
                    # Divide by 1000 for equivalence with initial implementation.
                    # Without a factor of 1/1000, the VB term hurts the MSE term.
                    terms["vb"] *= self.num_timesteps / 1000.0

            target = {
                ModelMeanType.PREVIOUS_X: self.q_posterior_mean_variance(
                    x_start=x_start, x_t=x_t, t=t
                )[0],
                ModelMeanType.START_X: x_start,
                ModelMeanType.EPSILON: noise,
                }[self.model_mean_type]

            assert model_output.shape == target.shape == x_start.shape
            terms["mse"] = mean_flat((target - model_output) ** 2)
            if "vb" in terms:
                terms["loss"] = terms["mse"] + terms["vb"]
            else:
                terms["loss"] = terms["mse"]
        else:
            raise NotImplementedError(self.loss_type)

        return terms

    def _prior_bpd(self, x_start):
        """
        Get the prior KL term for the variational lower-bound, measured in
        bits-per-dim.

        This term can't be optimized, as it only depends on the encoder.

        * x_start: the [N x C x ...] tensor of inputs.
        Returns:   a batch of [N] KL values (in bits), one per batch element.
        """
        batch_size = x_start.shape[0]
        t = torch.tensor([self.num_timesteps - 1] * batch_size, device=x_start.device)
        qt_mean, _, qt_log_variance = self.q_mean_variance(x_start, t)
        kl_prior = normal_kl(
            mean1=qt_mean, logvar1=qt_log_variance, mean2=0.0, logvar2=0.0
        )
        return mean_flat(kl_prior) / np.log(2.0)

    def calc_bpd_loop(self, model, x_start, clip_denoised=True, model_kwargs=None):
        """
        Compute the entire variational lower-bound, measured in bits-per-dim,
        as well as other related quantities.

        Arguments:
        * model:         the model to evaluate loss on.
        * x_start:       the [N x C x ...] tensor of inputs.
        * clip_denoised: if True, clip denoised samples.
        * model_kwargs:  if not None, a dict of extra keyword arguments to
                         pass to the model. This can be used for conditioning.

        Returns: a dict containing the following keys:
                 - total_bpd: the total variational lower-bound, per batch element.
                 - prior_bpd: the prior term in the lower-bound.
                 - vb: an [N x T] tensor of terms in the lower-bound.
                 - xstart_mse: an [N x T] tensor of x_0 MSEs for each timestep.
                 - mse: an [N x T] tensor of epsilon MSEs for each timestep.
        """
        device = x_start.device
        batch_size = x_start.shape[0]

        vb = []
        xstart_mse = []
        mse = []
        for t in list(range(self.num_timesteps))[::-1]:
            t_batch = torch.tensor([t] * batch_size, device=device)
            noise = torch.randn_like(x_start)
            x_t = self.q_sample(x_start=x_start, t=t_batch, noise=noise)
            # Calculate VLB term at the current timestep
            with torch.no_grad():
                out = self._vb_terms_bpd(
                    model,
                    x_start=x_start,
                    x_t=x_t,
                    t=t_batch,
                    clip_denoised=clip_denoised,
                    model_kwargs=model_kwargs,
                )
            vb.append(out["output"])
            xstart_mse.append(mean_flat((out["pred_xstart"] - x_start) ** 2))
            eps = self._predict_eps_from_xstart(x_t, t_batch, out["pred_xstart"])
            mse.append(mean_flat((eps - noise) ** 2))

        vb = torch.stack(vb, dim=1)
        xstart_mse = torch.stack(xstart_mse, dim=1)
        mse = torch.stack(mse, dim=1)

        prior_bpd = self._prior_bpd(x_start)
        total_bpd = vb.sum(dim=1) + prior_bpd
        return {
            "total_bpd": total_bpd,
            "prior_bpd": prior_bpd,
            "vb": vb,
            "xstart_mse": xstart_mse,
            "mse": mse,
        }


## Respace

In [ ]:
def space_timesteps(num_timesteps, section_counts):
    """
    Create a list of timesteps to use from an original diffusion process,
    given the number of timesteps we want to take from equally-sized portions
    of the original process.

    For example, if there are 300 timesteps and the section counts are [10,15,20]
    then the first 100 timesteps are strided to be 10 timesteps, the second 100
    are strided to be 15 timesteps, and the final 100 are strided to be 20.

    If the stride is a string starting with "ddim", then the fixed striding
    from the DDIM paper is used, and only one section is allowed.

    Argumebts:
    * num_timesteps:  the number of diffusion steps in the original
                      process to divide up.
    * section_counts: either a list of numbers, or a string containing
                      comma-separated numbers, indicating the step count
                      per section. As a special case, use "ddimN" where N
                      is a number of steps to use the striding from the
                      DDIM paper.
    Returns:          A set of diffusion steps from the original process to use.
    """
    if isinstance(section_counts, str):
        if section_counts.startswith("ddim"):
            desired_count = int(section_counts[len("ddim") :])
            for i in range(1, num_timesteps):
                if len(range(0, num_timesteps, i)) == desired_count:
                    return set(range(0, num_timesteps, i))
            raise ValueError(
                f"cannot create exactly {num_timesteps} steps with an integer stride"
            )
        section_counts = [int(x) for x in section_counts.split(",")]

    size_per  = num_timesteps // len(section_counts)
    extra     = num_timesteps % len(section_counts)
    start_idx = 0
    all_steps = []
    for i, section_count in enumerate(section_counts):
        size = size_per + (1 if i < extra else 0)
        if size < section_count:
            raise ValueError(
                f"cannot divide section of {size} steps into {section_count}"
            )
        if section_count <= 1:
            frac_stride = 1
        else:
            frac_stride = (size - 1) / (section_count - 1)
        cur_idx = 0.0
        taken_steps = []
        for _ in range(section_count):
            taken_steps.append(start_idx + round(cur_idx))
            cur_idx += frac_stride
        all_steps += taken_steps
        start_idx += size
    return set(all_steps)


class SpacedDiffusion(GaussianDiffusion):
    """
    A diffusion process which can skip steps in a base diffusion process.

    Arguments:
    * use_timesteps: a collection (sequence or set) of timesteps from the
                     original diffusion process to retain.
    * kwargs:        the kwargs to create the base diffusion process.
    """

    def __init__(self, use_timesteps, **kwargs):
        self.use_timesteps      = set(use_timesteps)
        self.timestep_map       = []
        self.original_num_steps = len(kwargs["betas"])

        base_diffusion = GaussianDiffusion(**kwargs)  # pylint: disable=missing-kwoa
        last_alpha_cumprod = 1.0
        new_betas = []
        for i, alpha_cumprod in enumerate(base_diffusion.alphas_cumprod):
            if i in self.use_timesteps:
                new_betas.append(1 - alpha_cumprod / last_alpha_cumprod)
                last_alpha_cumprod = alpha_cumprod
                self.timestep_map.append(i)
        kwargs["betas"] = np.array(new_betas)
        super().__init__(**kwargs)

    def p_mean_variance(
        self, model, *args, **kwargs
    ):  # pylint: disable=signature-differs
        return super().p_mean_variance(self._wrap_model(model), *args, **kwargs)

    def training_losses(
        self, model, *args, **kwargs
    ):  # pylint: disable=signature-differs
        return super().training_losses(self._wrap_model(model), *args, **kwargs)

    def condition_mean(self, cond_fn, *args, **kwargs):
        return super().condition_mean(self._wrap_model(cond_fn), *args, **kwargs)

    def condition_score(self, cond_fn, *args, **kwargs):
        return super().condition_score(self._wrap_model(cond_fn), *args, **kwargs)

    def _wrap_model(self, model):
        if isinstance(model, _WrappedModel):
            return model
        return _WrappedModel(
            model, self.timestep_map, self.rescale_timesteps, self.original_num_steps
        )

    def _scale_timesteps(self, t):
        # Scaling is done by the wrapped model.
        return t


class _WrappedModel:
    def __init__(self, model, timestep_map, rescale_timesteps, original_num_steps):
        self.model              = model
        self.timestep_map       = timestep_map
        self.rescale_timesteps  = rescale_timesteps
        self.original_num_steps = original_num_steps

    def __call__(self, x, ts, **kwargs):
        map_tensor = torch.tensor(self.timestep_map, device=ts.device, dtype=ts.dtype)
        new_ts     = map_tensor[ts]
        if self.rescale_timesteps:
            new_ts = new_ts.float() * (1000.0 / self.original_num_steps)

        return self.model(x, new_ts, **kwargs)

## Logger

In [ ]:
class Logger(object):

    def __init__(self):
        self.name2val = defaultdict(float)
        self.name2cnt = defaultdict(int)

    def logkv(self, key, val):
        self.name2val[key] = val

    def logkv_mean(self, key, val):
        oldval, cnt = self.name2val[key], self.name2cnt[key]
        self.name2val[key] = oldval * cnt / (cnt + 1) + val / (cnt + 1)
        self.name2cnt[key] = cnt + 1

    def dumpkvs(self):
        d   = self.name2val
        out = d.copy()        # Return the dict for unit testing purposes
        self.name2val.clear()
        self.name2cnt.clear()
        return out

    def printlog3(self, num_classes):
        if num_classes >= 10:
            k = 5
        else:
            k = num_classes // 2
        print(f'epoch: {self.name2val["epoch"]} step: {self.name2val["step"]}', end=' ')
        print(f'images: {self.name2val["images_processed"]}',   end=' | ')
        print(f'trainLoss: {self.name2val["train_loss"] :.6f}', end=' | ')
        print(f'trainAcc1: {self.name2val["train_acc1"] :.6f}', end=' | ')
        print(f'trainAcc{k}: {self.name2val[f"train_acc{k}"] :.6f}', end='')
        if "val_loss" in self.name2val.keys():
            print(f' | valLoss: {self.name2val["val_loss"] :.6f}', end=' | ')
            print(f'valAcc1: {self.name2val["val_acc1"] :.6f}', end=' | ')
            print(f'valAcc{k}: {self.name2val[f"val_acc{k}"] :.6f}')
        else:
            print('\n')

    def logWandB(self, num_classes):
        if num_classes >= 10:
            k = 5
        else:
            k = num_classes // 2
        train_mult_name = f"train_acc{k}"
        val_mult_name   = f"val_acc{k}"

        wandb_dict = {
            "epoch":         self.name2val["epoch"],
            "step":          self.name2val["step"],
            "images":        self.name2val["images_processed"],
            "train_loss":    self.name2val["train_loss"],
            "train_loss_Q0": self.name2val["train_loss_q0"],
            "train_loss_Q1": self.name2val["train_loss_q1"],
            "train_loss_Q2": self.name2val["train_loss_q2"],
            "train_loss_Q3": self.name2val["train_loss_q3"],
            "train_acc1":    self.name2val["train_acc1"],
            "train_acc1_Q0": self.name2val["train_acc1_q0"],
            "train_acc1_Q1": self.name2val["train_acc1_q1"],
            "train_acc1_Q2": self.name2val["train_acc1_q2"],
            "train_acc1_Q3": self.name2val["train_acc1_q3"]
        }
        if train_mult_name in self.name2val.keys():
            wandb_dict[f"train_acc{k}"]    = self.name2val[f"train_acc{k}"]
            wandb_dict[f"train_acc{k}_Q0"] = self.name2val[f"train_acc{k}_q0"]
            wandb_dict[f"train_acc{k}_Q1"] = self.name2val[f"train_acc{k}_q1"]
            wandb_dict[f"train_acc{k}_Q2"] = self.name2val[f"train_acc{k}_q2"]
            wandb_dict[f"train_acc{k}_Q3"] = self.name2val[f"train_acc{k}_q3"]

        if "val_loss" in self.name2val.keys():
            wandb_dict["val_loss"]       = self.name2val["val_loss"]
            wandb_dict["val_loss_Q0"]    = self.name2val["val_loss_q0"]
            wandb_dict["val_loss_Q1"]    = self.name2val["val_loss_q1"]
            wandb_dict["val_loss_Q2"]    = self.name2val["val_loss_q2"]
            wandb_dict["val_loss_Q3"]    = self.name2val["val_loss_q3"]
            wandb_dict["val_acc1"]       = self.name2val["val_acc1"]
            wandb_dict["val_acc1_Q0"]    = self.name2val["val_acc1_q0"]
            wandb_dict["val_acc1_Q1"]    = self.name2val["val_acc1_q1"]
            wandb_dict["val_acc1_Q2"]    = self.name2val["val_acc1_q2"]
            wandb_dict["val_acc1_Q3"]    = self.name2val["val_acc1_q3"]
        if val_mult_name in self.name2val.keys():
            wandb_dict[f"val_acc{k}"]    = self.name2val[f"val_acc{k}"]
            wandb_dict[f"val_acc{k}_Q0"] = self.name2val[f"val_acc{k}_q0"]
            wandb_dict[f"val_acc{k}_Q1"] = self.name2val[f"val_acc{k}_q1"]
            wandb_dict[f"val_acc{k}_Q2"] = self.name2val[f"val_acc{k}_q2"]
            wandb_dict[f"val_acc{k}_Q3"] = self.name2val[f"val_acc{k}_q3"]

        try:
            # Log metrics to Weights and Biases .........................
            wandb.log(wandb_dict)
        except Exception as ex:
            print(f'An exception of type {type(ex).__name__} occurred. Arguments:\n{ex.args!r}')

## U-Net Encoder to Model the Classifier

In [ ]:
class Upsample(nn.Module):
    """
    An upsampling layer with an optional convolution.

    Arguments:
    * channels: channels in the inputs and outputs.
    * use_conv: a bool determining if a convolution is applied.
    * dims:     determines if the signal is 1D, 2D, or 3D. If 3D, then
                upsampling occurs in the inner-two dimensions.
    """

    def __init__(self, channels, use_conv, dims=2, out_channels=None):
        super().__init__()
        self.channels     = channels
        self.out_channels = out_channels or channels
        self.use_conv     = use_conv
        self.dims         = dims
        if use_conv:
            self.conv = conv_nd(dims, self.channels, self.out_channels, 3, padding=1)

    def forward(self, x):
        assert x.shape[1] == self.channels
        if self.dims == 3:
            x = F.interpolate(
                x, (x.shape[2], x.shape[3] * 2, x.shape[4] * 2), mode="nearest"
            )
        else:
            x = F.interpolate(x, scale_factor=2, mode="nearest")
        if self.use_conv:
            x = self.conv(x)
        return x


class Downsample(nn.Module):
    """
    A downsampling layer with an optional convolution.

    Attributes:
    * channels: channels in the inputs and outputs.
    * use_conv: a bool determining if a convolution is applied.
    * dims:     determines if the signal is 1D, 2D, or 3D. If 3D, then
                downsampling occurs in the inner-two dimensions.
    """

    def __init__(self, channels, use_conv, dims=2, out_channels=None):
        super().__init__()
        self.channels     = channels
        self.out_channels = out_channels or channels
        self.use_conv     = use_conv
        self.dims         = dims
        stride            = 2 if dims != 3 else (1, 2, 2)
        if use_conv:
            self.op = conv_nd(
                dims, self.channels, self.out_channels, 3, stride=stride, padding=1
            )
        else:
            assert self.channels == self.out_channels
            self.op = avg_pool_nd(dims, kernel_size=stride, stride=stride)

    def forward(self, x):
        assert x.shape[1] == self.channels
        return self.op(x)


class ResBlock(TimestepBlock):
    """
    A residual block that can optionally changes the number of channels.

    Arguments:
    * channels:       the number of input channels.
    * emb_channels:   the number of timestep embedding channels.
    * dropout:        the rate of dropout.
    * out_channels:   if specified, the number of out channels.
    * use_conv:       if True and out_channels is specified, use a spatial
                      convolution instead of a smaller 1x1 convolution 
                      to change the channels in the skip connection.
    * dims:           determines if the signal is 1D, 2D, or 3D.
    * use_checkpoint: if True, use gradient checkpointing on this module.
    * up:             if True, use this block for upsampling.
    * down:           if True, use this block for downsampling.
    """

    def __init__(
        self,
        channels,
        emb_channels,
        dropout,
        out_channels         = None,
        use_conv             = False,
        use_scale_shift_norm = False,
        dims                 = 2,
        use_checkpoint       = False,
        up                   = False,
        down                 = False,
    ):
        super().__init__()
        self.channels             = channels
        self.emb_channels         = emb_channels
        self.dropout              = dropout
        self.out_channels         = out_channels or channels
        self.use_conv             = use_conv
        self.use_checkpoint       = use_checkpoint
        self.use_scale_shift_norm = use_scale_shift_norm

        self.in_layers = nn.Sequential(
            normalization(channels),
            nn.SiLU(),
            conv_nd(dims, channels, self.out_channels, 3, padding=1),
        )

        self.updown = up or down

        if up:
            self.h_upd = Upsample(channels, False, dims)
            self.x_upd = Upsample(channels, False, dims)
        elif down:
            self.h_upd = Downsample(channels, False, dims)
            self.x_upd = Downsample(channels, False, dims)
        else:
            self.h_upd = self.x_upd = nn.Identity()

        self.emb_layers = nn.Sequential(
            nn.SiLU(),
            linear(
                emb_channels,
                2 * self.out_channels if use_scale_shift_norm else self.out_channels,
            ),
        )
        self.out_layers = nn.Sequential(
            normalization(self.out_channels),
            nn.SiLU(),
            nn.Dropout(p=dropout),
            zero_module(
                conv_nd(dims, self.out_channels, self.out_channels, 3, padding=1)
            ),
        )

        if self.out_channels == channels:
            self.skip_connection = nn.Identity()
        elif use_conv:
            self.skip_connection = conv_nd(
                dims, channels, self.out_channels, 3, padding=1
            )
        else:
            self.skip_connection = conv_nd(dims, channels, self.out_channels, 1)

    def forward(self, x, emb):
        """
        Apply the block to a tensor, conditioned on a timestep embedding.

        Arguments:
        * x:       an [N x C x ...] tensor of features.
        * emb:     an [N x emb_channels] tensor of timestep embeddings.
        Returns:   an [N x C x ...] tensor of outputs.
        """
        return checkpoint(
            self._forward, (x, emb), self.parameters(), self.use_checkpoint
        )

    def _forward(self, x, emb):
        if self.updown:
            in_rest, in_conv = self.in_layers[:-1], self.in_layers[-1]
            h = in_rest(x)
            h = self.h_upd(h)
            x = self.x_upd(x)
            h = in_conv(h)
        else:
            h = self.in_layers(x)
        emb_out = self.emb_layers(emb).type(h.dtype)
        while len(emb_out.shape) < len(h.shape):
            emb_out = emb_out[..., None]
        if self.use_scale_shift_norm:
            out_norm, out_rest = self.out_layers[0], self.out_layers[1:]
            scale, shift = torch.chunk(emb_out, 2, dim=1)
            h = out_norm(h) * (1 + scale) + shift
            h = out_rest(h)
        else:
            h = h + emb_out
            h = self.out_layers(h)
        return self.skip_connection(x) + h


In [ ]:
class EncoderUNetModel(nn.Module):
    """
    The U-Net encoder model with attention and timestep embedding.
    """
    def __init__(
        self,
        image_size,
        in_channels,
        model_channels,
        out_channels,
        num_res_blocks,
        attention_resolutions,
        dropout                 = 0,
        channel_mult            = (1, 2, 4, 8),
        conv_resample           = True,
        dims                    = 2,
        use_checkpoint          = False,
        use_fp16                = False,
        num_heads               = 1,
        num_head_channels       = -1,
        num_heads_upsample      = -1,
        use_scale_shift_norm    = False,
        resblock_updown         = False,
        use_new_attention_order = False,
        pool                    = "adaptive",
        ):
        super().__init__()

        if num_heads_upsample == -1:
            num_heads_upsample = num_heads

        self.in_channels           = in_channels
        self.model_channels        = model_channels
        self.out_channels          = out_channels
        self.num_res_blocks        = num_res_blocks
        self.attention_resolutions = attention_resolutions
        self.dropout               = dropout
        self.channel_mult          = channel_mult
        self.conv_resample         = conv_resample
        self.use_checkpoint        = use_checkpoint
        self.dtype                 = torch.float16 if use_fp16 else torch.float32
        self.num_heads             = num_heads
        self.num_head_channels     = num_head_channels
        self.num_heads_upsample    = num_heads_upsample

        time_embed_dim  = model_channels * 4
        self.time_embed = nn.Sequential(
            linear(model_channels, time_embed_dim),
            nn.SiLU(),
            linear(time_embed_dim, time_embed_dim),
        )

        ch = int(channel_mult[0] * model_channels)
        self.input_blocks = nn.ModuleList(
            [TimestepEmbedSequential(conv_nd(dims, in_channels, ch, 3, padding=1))]
        )
        self._feature_size = ch
        input_block_chans = [ch]
        ds = 1
        for level, mult in enumerate(channel_mult):
            for _ in range(num_res_blocks):
                layers = [
                    ResBlock(
                        ch,
                        time_embed_dim,
                        dropout,
                        out_channels         = int(mult * model_channels),
                        dims                 = dims,
                        use_checkpoint       = use_checkpoint,
                        use_scale_shift_norm = use_scale_shift_norm,
                    )
                ]
                ch = int(mult * model_channels)
                if ds in attention_resolutions:
                    layers.append(
                        AttentionBlock(
                            ch,
                            use_checkpoint          = use_checkpoint,
                            num_heads               = num_heads,
                            num_head_channels       = num_head_channels,
                            use_new_attention_order = use_new_attention_order,
                        )
                    )
                self.input_blocks.append(TimestepEmbedSequential(*layers))
                self._feature_size += ch
                input_block_chans.append(ch)
            if level != len(channel_mult) - 1:
                out_ch = ch
                self.input_blocks.append(
                    TimestepEmbedSequential(
                        ResBlock(
                            ch,
                            time_embed_dim,
                            dropout,
                            out_channels         = out_ch,
                            dims                 = dims,
                            use_checkpoint       = use_checkpoint,
                            use_scale_shift_norm = use_scale_shift_norm,
                            down                 = True,
                        )
                        if resblock_updown
                        else Downsample(
                            ch,
                            conv_resample,
                            dims           = dims,
                            out_channels   = out_ch,
                        )
                    )
                )
                ch  = out_ch
                input_block_chans.append(ch)
                ds *= 2
                self._feature_size += ch

        self.middle_block = TimestepEmbedSequential(
            ResBlock(
                ch,
                time_embed_dim,
                dropout,
                dims                 = dims,
                use_checkpoint       = use_checkpoint,
                use_scale_shift_norm = use_scale_shift_norm,
            ),
            AttentionBlock(
                ch,
                use_checkpoint          = use_checkpoint,
                num_heads               = num_heads,
                num_head_channels       = num_head_channels,
                use_new_attention_order = use_new_attention_order,
            ),
            ResBlock(
                ch,
                time_embed_dim,
                dropout,
                dims                 = dims,
                use_checkpoint       = use_checkpoint,
                use_scale_shift_norm = use_scale_shift_norm,
            ),
        )
        self._feature_size += ch
        self.pool = pool
        if pool == "adaptive":
            self.out = nn.Sequential(
                normalization(ch),
                nn.SiLU(),
                nn.AdaptiveAvgPool2d((1, 1)),
                zero_module(conv_nd(dims, ch, out_channels, 1)),
                nn.Flatten(),
            )
        elif pool == "attention":
            assert num_head_channels != -1
            self.out = nn.Sequential(
                normalization(ch),
                nn.SiLU(),
                AttentionPool2d(
                    (image_size // ds), ch, num_head_channels, out_channels
                ),
            )
        elif pool == "spatial":
            self.out = nn.Sequential(
                nn.Linear(self._feature_size, 2048),
                nn.ReLU(),
                nn.Linear(2048, self.out_channels),
            )
        elif pool == "spatial_v2":
            self.out = nn.Sequential(
                nn.Linear(self._feature_size, 2048),
                normalization(2048),
                nn.SiLU(),
                nn.Linear(2048, self.out_channels),
            )
        else:
            raise NotImplementedError(f"Unexpected {pool} pooling")

    def convert_to_fp16(self):
        """
        Convert the torso of the model to float16.
        """
        self.input_blocks.apply(convert_module_to_f16)
        self.middle_block.apply(convert_module_to_f16)

    def convert_to_fp32(self):
        """
        Convert the torso of the model to float32.
        """
        self.input_blocks.apply(convert_module_to_f32)
        self.middle_block.apply(convert_module_to_f32)

    def forward(self, x, timesteps):
        """
        Apply the model to an input batch.

        :param x: an [N x C x ...] Tensor of inputs.
        :param timesteps: a 1-D batch of timesteps.
        :return: an [N x K] Tensor of outputs.
        """

        ts_emb = timestep_embedding(timesteps, self.model_channels)

        emb = self.time_embed(ts_emb)

        results = []
        h       = x.type(self.dtype)
        for module in self.input_blocks:
            h = module(h, emb)
            if self.pool.startswith("spatial"):
                results.append(h.type(x.dtype).mean(dim=(2, 3)))
        h = self.middle_block(h, emb)
        if self.pool.startswith("spatial"):
            results.append(h.type(x.dtype).mean(dim=(2, 3)))
            h = torch.cat(results, axis=-1)
            return self.out(h)
        else:
            h = h.type(x.dtype)
            return self.out(h)


# U-Net Used by the Diffusion Model

In [ ]:
class UNetModel(nn.Module):
    """
    The UNet model with attention and timestep embedding.

    Attributes:
    * in_channels:    channels in the input Tensor.
    * model_channels: base channel count for the model.
    * out_channels:   channels in the output Tensor.
    * num_res_blocks: number of residual blocks per resolution.
    * attention_resolutions: a collection of resolutions at which
        attention will take place. May be a set, list, or tuple.
        For example, if this contains 4, then at 4x downsampling, attention
        will be used.
    * dropout:        the dropout probability.
    * channel_mult:   channel multiplier for each level of the UNet.
    * conv_resample:  if True, use learned convolutions for upsampling 
                      and downsampling.
    * dims:           determines if the signal is 1D, 2D, or 3D.
    * num_classes:    if specified (as an int), then this model will be
                      class-conditional with `num_classes` classes.
    * use_checkpoint: use gradient checkpointing to reduce memory usage.
    * num_heads:      the  number of attention heads in each attention layer.
    * num_heads_channels:     if specified, ignore num_heads and instead use
                               a fixed channel width per attention head.
    * num_heads_upsample:      works with num_heads to set a different number
                               of heads for upsampling. Deprecated.
    * use_scale_shift_norm:    use a FiLM-like conditioning mechanism.
    * resblock_updown:         use residual blocks for up/downsampling.
    * use_new_attention_order: use a different attention pattern for potentially
                               increased efficiency.
    """

    def __init__(
        self,
        image_size,
        in_channels,
        model_channels,
        out_channels,
        num_res_blocks,
        attention_resolutions,
        dropout                 = 0,
        channel_mult            = (1, 2, 4, 8),
        conv_resample           = True,
        dims                    = 2,
        num_classes             = None,
        use_checkpoint          = False,
        use_fp16                = False,
        num_heads               = 1,
        num_head_channels       = -1,
        num_heads_upsample      = -1,
        use_scale_shift_norm    = False,
        resblock_updown         = False,
        use_new_attention_order = False,
    ):
        super().__init__()

        if num_heads_upsample == -1:
            num_heads_upsample = num_heads

        self.image_size         = image_size
        self.in_channels        = in_channels
        self.model_channels     = model_channels
        self.out_channels       = out_channels
        self.num_res_blocks     = num_res_blocks
        self.attention_resolutions = attention_resolutions
        self.dropout            = dropout
        self.channel_mult       = channel_mult
        self.conv_resample      = conv_resample
        self.num_classes        = num_classes
        self.use_checkpoint     = use_checkpoint
        self.dtype              = torch.float16 if use_fp16 else torch.float32
        self.num_heads          = num_heads
        self.num_head_channels  = num_head_channels
        self.num_heads_upsample = num_heads_upsample

        time_embed_dim = model_channels * 4
        self.time_embed = nn.Sequential(
            linear(model_channels, time_embed_dim),
            nn.SiLU(),
            linear(time_embed_dim, time_embed_dim),
        )

        if self.num_classes is not None:
            self.label_emb = nn.Embedding(num_classes, time_embed_dim)

        ch                = input_ch = int(channel_mult[0] * model_channels)
        self.input_blocks = nn.ModuleList(
            [TimestepEmbedSequential(conv_nd(dims, in_channels, ch, 3, padding=1))]
        )
        self._feature_size = ch
        input_block_chans = [ch]
        ds = 1
        for level, mult in enumerate(channel_mult):
            for _ in range(num_res_blocks):
                layers = [
                    ResBlock(
                        ch,
                        time_embed_dim,
                        dropout,
                        out_channels=int(mult * model_channels),
                        dims=dims,
                        use_checkpoint=use_checkpoint,
                        use_scale_shift_norm=use_scale_shift_norm,
                    )
                ]
                ch = int(mult * model_channels)
                if ds in attention_resolutions:
                    layers.append(
                        AttentionBlock(
                            ch,
                            use_checkpoint=use_checkpoint,
                            num_heads=num_heads,
                            num_head_channels=num_head_channels,
                            use_new_attention_order=use_new_attention_order,
                        )
                    )
                self.input_blocks.append(TimestepEmbedSequential(*layers))
                self._feature_size += ch
                input_block_chans.append(ch)
            if level != len(channel_mult) - 1:
                out_ch = ch
                self.input_blocks.append(
                    TimestepEmbedSequential(
                        ResBlock(
                            ch,
                            time_embed_dim,
                            dropout,
                            out_channels=out_ch,
                            dims=dims,
                            use_checkpoint=use_checkpoint,
                            use_scale_shift_norm=use_scale_shift_norm,
                            down=True,
                        )
                        if resblock_updown
                        else Downsample(
                            ch, conv_resample, dims=dims, out_channels=out_ch
                        )
                    )
                )
                ch = out_ch
                input_block_chans.append(ch)
                ds *= 2
                self._feature_size += ch

        self.middle_block = TimestepEmbedSequential(
            ResBlock(
                ch,
                time_embed_dim,
                dropout,
                dims=dims,
                use_checkpoint=use_checkpoint,
                use_scale_shift_norm=use_scale_shift_norm,
            ),
            AttentionBlock(
                ch,
                use_checkpoint=use_checkpoint,
                num_heads=num_heads,
                num_head_channels=num_head_channels,
                use_new_attention_order=use_new_attention_order,
            ),
            ResBlock(
                ch,
                time_embed_dim,
                dropout,
                dims=dims,
                use_checkpoint=use_checkpoint,
                use_scale_shift_norm=use_scale_shift_norm,
            ),
        )
        self._feature_size += ch

        self.output_blocks = nn.ModuleList([])
        for level, mult in list(enumerate(channel_mult))[::-1]:
            for i in range(num_res_blocks + 1):
                ich = input_block_chans.pop()
                layers = [
                    ResBlock(
                        ch + ich,
                        time_embed_dim,
                        dropout,
                        out_channels=int(model_channels * mult),
                        dims=dims,
                        use_checkpoint=use_checkpoint,
                        use_scale_shift_norm=use_scale_shift_norm,
                    )
                ]
                ch = int(model_channels * mult)
                if ds in attention_resolutions:
                    layers.append(
                        AttentionBlock(
                            ch,
                            use_checkpoint=use_checkpoint,
                            num_heads=num_heads_upsample,
                            num_head_channels=num_head_channels,
                            use_new_attention_order=use_new_attention_order,
                        )
                    )
                if level and i == num_res_blocks:
                    out_ch = ch
                    layers.append(
                        ResBlock(
                            ch,
                            time_embed_dim,
                            dropout,
                            out_channels=out_ch,
                            dims=dims,
                            use_checkpoint=use_checkpoint,
                            use_scale_shift_norm=use_scale_shift_norm,
                            up=True,
                        )
                        if resblock_updown
                        else Upsample(ch, conv_resample, dims=dims, out_channels=out_ch)
                    )
                    ds //= 2
                self.output_blocks.append(TimestepEmbedSequential(*layers))
                self._feature_size += ch

        self.out = nn.Sequential(
            normalization(ch),
            nn.SiLU(),
            zero_module(conv_nd(dims, input_ch, out_channels, 3, padding=1)),
        )

    def convert_to_fp16(self):
        """
        Convert the torso of the model to float16.
        """
        self.input_blocks.apply(convert_module_to_f16)
        self.middle_block.apply(convert_module_to_f16)
        self.output_blocks.apply(convert_module_to_f16)

    def convert_to_fp32(self):
        """
        Convert the torso of the model to float32.
        """
        self.input_blocks.apply(convert_module_to_f32)
        self.middle_block.apply(convert_module_to_f32)
        self.output_blocks.apply(convert_module_to_f32)

    def forward(self, x, timesteps, y=None):
        """
        Apply the model to an input batch.

        * x:         an [N x C x ...] Tensor of inputs.
        * timesteps: a 1-D batch of timesteps.
        * y:         an [N] Tensor of labels, if class-conditional.
        Returns:     an [N x C x ...] Tensor of outputs.
        """
        assert (y is not None) == (
            self.num_classes is not None
        ), "must specify y if and only if the model is class-conditional"

        hs = []
        emb = self.time_embed(timestep_embedding(timesteps, self.model_channels))

        if self.num_classes is not None:
            assert y.shape == (x.shape[0],)
            emb = emb + self.label_emb(y)

        h = x.type(self.dtype)
        for module in self.input_blocks:
            h = module(h, emb)
            hs.append(h)
        h = self.middle_block(h, emb)
        for module in self.output_blocks:
            h = torch.cat([h, hs.pop()], dim=1)
            h = module(h, emb)
        h = h.type(x.dtype)
        return self.out(h)


# Class to manage classifier-guided sampling from the diffusion model

In [ ]:
class SamplingSession:

    def __init__(
        self,
        logger,
        config,
        ):
        self.config            = config
        self.logger            = logger
        self.model             = None
        self.diffusion         = None
        self.num_classes       = config.data.num_classes
        self.batch_size        = config.sampling.batch_size
        self.num_samples       = config.sampling.num_samples
        self.device            = config.sampling.device
        self.use_ddim          = config.sampling.use_ddim
        self.clip_denoised     = config.sampling.clip_denoised
        self.image_size        = config.data.image_size
        self.num_classes       = config.data.num_classes
        self.images            = None
        self.labels            = None

    # ================================================================== OKAT
    def restore_checkpoint_model(self, model_file, model, device):
        '''
        Restore a model checkpoint from file.
        '''
        if model_file is not None:
            if os.path.isfile(model_file) == True:
                loaded_state = torch.load(model_file, map_location='cpu', weights_only=True)
                model.load_state_dict(loaded_state['model'], strict=False)
                model = model.to(device)
                print(f'[INFO] Loaded model checkpoint from {model_file}!')
            else:
                print(f'[ERROR] Checkpoint {model_file} does not exist!')


    # ==================================================================
    def guided_sampling(self):
        '''
        Classifier-guided sampling from a diffusion model.
        '''

        print("[INFO] create the diffusion model and load the weights from file ...")

        self.create_model_and_diffusion()

        self.model.to(self.config.sampling.device)

        self.restore_checkpoint_model(
            self.config.sampling.model_file,
            self.model,
            self.config.sampling.device,
        )

        if self.config.training.use_fp16:
            self.model.convert_to_fp16()
        self.model.eval()

        print("[INFO] create the classifier and load the weights from file ...")

        self.classifier = self.create_classifier(
            self.config.data.image_size,
            self.config.data.num_classes,
            self.config.classifier.use_fp16,
            self.config.classifier.width,
            self.config.classifier.depth,
            self.config.classifier.attention_resolutions,
            self.config.classifier.use_scale_shift_norm,
            self.config.classifier.resblock_updown,
            self.config.classifier.pool,
        )

        self.restore_checkpoint_model(
            self.config.sampling.classifier_file,
            self.classifier,
            self.config.sampling.device,
        )

        if self.config.classifier.use_fp16:
            self.classifier.convert_to_fp16()

        self.classifier.eval()

        def cond_fn(x, t, y=None):
            assert y is not None
            with torch.enable_grad():
                x_in      = x.detach().requires_grad_(True)

                logits    = self.classifier(x_in, t)
                log_probs = F.log_softmax(logits, dim=-1)
                selected  = log_probs[range(len(logits)), y.view(-1)]
                return torch.autograd.grad(selected.sum(), x_in)[0] * self.config.classifier.scale

        def model_fn(x, t, y=None):
            assert y is not None
            return self.model(x, t, y if self.config.model.class_cond else None)

        print("[INFO] sampling ...")
        all_images = []
        all_labels = []

        num_iterations = self.num_samples // self.batch_size
        for n in range(num_iterations):
            model_kwargs = {}
            # Create a tensor filled with all class IDs
            classes = torch.randint(
                low    = 0,
                high   = self.num_classes,
                size   = (self.batch_size,),
                device = self.device,
            )
            # Get 'batch_size' sample from the diffusion model
            model_kwargs["y"] = classes
            sample_fn = (
                self.diffusion.p_sample_loop if not self.use_ddim else self.diffusion.ddim_sample_loop
            )
            samples = sample_fn(
                model_fn,
                (self.batch_size, 3, self.image_size, self.image_size),
                clip_denoised = self.clip_denoised,
                model_kwargs  = model_kwargs,
                cond_fn       = cond_fn,
                device        = self.device,
            )
            # Convert samples to the 0..255 range
            samples = ((samples + 1) * 127.5).clamp(0, 255).to(torch.uint8)
            samples = samples.permute(0, 2, 3, 1)
            samples = samples.contiguous()

            # Save samples in array 'all_images' and 'all_labels'
            all_images.extend([sample.cpu().numpy() for sample in samples])
            all_labels.extend(classes.cpu().numpy())
            print(f"[INFO] created {(n+1) * self.batch_size} samples")
            print(f'[INFO] image labels: {all_labels}')

        print(f'[INFO] generated images: list of {len(all_images)} elements with shape {all_images[0].shape}')

        self.images = np.stack(all_images, axis=0)
        self.images = self.images[: self.num_samples]
        self.labels = np.asarray(all_labels)
        self.labels = self.labels[: self.num_samples]

        shape_str = "x".join([str(x) for x in self.images.shape])
        fname = os.path.join(
            self.config.experiment.root_dir,
            self.config.experiment.results_dir,
            self.config.experiment.experiment_name,
            f"generated_images_{self.config.experiment.experiment_name}.npz"
        )
        print(f"[INFO] saving images ({self.images.shape}) and labels ({self.labels.shape}) to {fname}")
        np.savez(fname, self.images, self.labels)

        print("[INFO] sampling complete")

    # ==================================================================
    def create_model_and_diffusion(self):

        self.model = self.create_model(
            self.config.data.image_size,
            self.config.model.num_channels,
            self.config.model.num_res_blocks,
            self.config.model.channel_mult,
            self.config.training.learn_sigma,
            self.config.model.class_cond,
            self.config.training.use_gradient_checkpointing,
            self.config.model.attention_resolutions,
            self.config.model.num_heads,
            self.config.model.num_head_channels,
            self.config.model.num_heads_upsample,
            self.config.model.use_scale_shift_norm,
            self.config.model.dropout,
            self.config.model.resblock_updown,
            self.config.training.use_fp16,
            self.config.model.use_new_attention_order,
            self.config.data.num_classes,
        )

        self.diffusion = self.create_gaussian_diffusion(
            steps                  = self.config.training.diffusion_steps,
            learn_sigma            = self.config.training.learn_sigma,
            noise_schedule         = self.config.training.noise_schedule,
            use_kl                 = self.config.training.use_kl,
            predict_xstart         = self.config.training.predict_xstart,
            rescale_timesteps      = self.config.training.rescale_timesteps,
            rescale_learned_sigmas = self.config.training.rescale_learned_sigmas,
            timestep_respacing     = self.config.training.timestep_respacing,
        )

    # ==================================================================
    def create_model(
        self,
        image_size,
        num_channels,
        num_res_blocks,
        channel_mult            = "",
        learn_sigma             = False,
        class_cond              = False,
        use_checkpoint          = False,
        attention_resolutions   = "16",
        num_heads               = 1,
        num_head_channels       = -1,
        num_heads_upsample      = -1,
        use_scale_shift_norm    = False,
        dropout                 = 0,
        resblock_updown         = False,
        use_fp16                = False,
        use_new_attention_order = False,
        num_classes             = 1000,
        ):
        if channel_mult == "":
            if image_size == 512:
                channel_mult = (0.5, 1, 1, 2, 2, 4, 4)
            elif image_size == 256:
                channel_mult = (1, 1, 2, 2, 4, 4)
            elif image_size == 128:
                channel_mult = (1, 1, 2, 3, 4)
            elif image_size == 64:
                channel_mult = (1, 2, 3, 4)
            else:
                raise ValueError(f"unsupported image size: {image_size} x {image_size}")
        else:
            channel_mult = tuple(int(ch_mult) for ch_mult in channel_mult.split(","))

        attention_ds = []
        for res in attention_resolutions.split(","):
            attention_ds.append(image_size // int(res))

        return UNetModel(
            image_size              = image_size,
            in_channels             = 3,
            model_channels          = num_channels,
            out_channels            = (3 if not learn_sigma else 6),
            num_res_blocks          = num_res_blocks,
            attention_resolutions   = tuple(attention_ds),
            dropout                 = dropout,
            channel_mult            = channel_mult,
            num_classes             = (num_classes if class_cond else None),
            use_checkpoint          = use_checkpoint,
            use_fp16                = use_fp16,
            num_heads               = num_heads,
            num_head_channels       = num_head_channels,
            num_heads_upsample      = num_heads_upsample,
            use_scale_shift_norm    = use_scale_shift_norm,
            resblock_updown         = resblock_updown,
            use_new_attention_order = use_new_attention_order,
        )

    # ==================================================================
    def create_gaussian_diffusion(
        self,
        steps                  = 1000,
        learn_sigma            = False,
        sigma_small            = False,
        noise_schedule         = "linear",
        use_kl                 = False,
        predict_xstart         = False,
        rescale_timesteps      = False,
        rescale_learned_sigmas = False,
        timestep_respacing     = "",
        ):
        betas = get_named_beta_schedule(noise_schedule, steps)
        if use_kl:
            loss_type = LossType.RESCALED_KL
        elif rescale_learned_sigmas:
            loss_type = LossType.RESCALED_MSE
        else:
            loss_type = LossType.MSE
        if not timestep_respacing:
            timestep_respacing = [steps]
        return SpacedDiffusion(
            use_timesteps   = space_timesteps(steps, timestep_respacing),
            betas           = betas,
            model_mean_type = (
                ModelMeanType.EPSILON if not predict_xstart else ModelMeanType.START_X
            ),
            model_var_type = (
                (
                    ModelVarType.FIXED_LARGE
                    if not sigma_small
                    else ModelVarType.FIXED_SMALL
                )
                if not learn_sigma
                else ModelVarType.LEARNED_RANGE
            ),
            loss_type         = loss_type,
            rescale_timesteps = rescale_timesteps,
        )

    # ==================================================================
    def create_classifier(
        self,
        image_size,
        num_classes,
        classifier_use_fp16,
        classifier_width,
        classifier_depth,
        classifier_attention_resolutions,
        classifier_use_scale_shift_norm,
        classifier_resblock_updown,
        classifier_pool,
        ):
        if image_size == 512:
            channel_mult = (0.5, 1, 1, 2, 2, 4, 4)
        elif image_size == 256:
            channel_mult = (1, 1, 2, 2, 4, 4)
        elif image_size == 128:
            channel_mult = (1, 1, 2, 3, 4)
        elif image_size == 64:
            channel_mult = (1, 2, 3, 4)
        else:
            raise ValueError(f"[WARN] unsupported image size {image_size} x {image_size}")

        attention_ds = []
        for res in classifier_attention_resolutions.split(","):
            attention_ds.append(image_size // int(res))

        return EncoderUNetModel(
            image_size            = image_size,
            in_channels           = 3,
            model_channels        = classifier_width,
            out_channels          = num_classes,
            num_res_blocks        = classifier_depth,
            attention_resolutions = tuple(attention_ds),
            channel_mult          = channel_mult,
            use_fp16              = classifier_use_fp16,
            num_head_channels     = 64,
            use_scale_shift_norm  = classifier_use_scale_shift_norm,
            resblock_updown       = classifier_resblock_updown,
            pool                  = classifier_pool,
        )


# Classifier Guided Sampling

For sampling from a 128x128 classifier-guided model, using 25 steps DDIM, use the following configuration:
* config.model.attention_resolutions      = "32,16,8"
* config.model.num_channels               = 256
* config.model.num_heads                  = 4
* config.model.num_res_blocks             = 2
* config.model.resblock_updown            = True
* config.model.use_scale_shift_norm       = True
* config.model.class_cond                 = True
* config.data.image_size                  = 128
* config.training.use_fp16                = True
* config.training.learn_sigma             = True
* config.classifier.attention_resolutions = "32,16,8"
* config.classifier.depth                 = 2
* config.classifier.width                 = 128
* config.classifier.pool                  = "attention"
* config.classifier.resblock_updown       = True
* config.classifier.use_scale_shift_norm  = True
* config.classifier.scale                 = 1.0
* config.classifier.use_fp16              = True
* config.sampling.batch_size              = 4
* config.sampling.num_samples             = 50000
* config.training.timestep_respacing      = "ddim25"
* config.sampling.use_ddim                = True

To sample for 250 timesteps without DDIM, use the previous configuration values with two modifications:
* config.training.timestep_respacing      = 250
* config.sampling.use_ddim                = False

In [ ]:
def run_sampling(config):

    logger = Logger()

    sampling_session = SamplingSession(
        logger = logger,
        config = config,
    )

    sampling_session.guided_sampling()
    return sampling_session, logger

## Login into Weights & Bias

In [ ]:
wandb.login()

## Track metadata and hyperparameters with Weights & Bias

Define the experiment: the hyperparameters, the dataset and model name. This information will be stored in a `config` dictionary.

In [ ]:
config_wandb = config

wandb.init(
    project = 'OUR_WANDB_PROJECT_ID',
    entity  = 'OUR_WANDB_ENTITY', 
    #id      = config.experiment.experiment_name, # TO RESUME LOGGING
    #resume  = 'allow',                           # TO RESUME LOGGING
)

In [ ]:
def find_firstN(arr: np.array, values: list, n: int):
	'''
	Looks for the first 'n' occurrences of each value from 'values'
	in the array 'arr'.
	Returns a dictionary where the keys are the values from 'values' and 
	the values are the indices of the position where each value occurs in array 'arr'.
	'''
	res = {}
	for v in values:
		indices = []
		i, found = 0, 0
		while found < n:
			try:
				if (arr[i] == v):
					indices.append(i)
					found += 1
			except IndexError:
				return None
			i += 1
		res[v] = indices
	return res

In [ ]:
indices = None
while indices is None:
    session, logger = run_sampling(config)

    print(f'images: {session.images.shape} labels: {session.labels.shape}')

    IMGS_PER_CLASS  = 5
    indices         = find_firstN(session.labels, [0,1,2,3], IMGS_PER_CLASS)
    if indices is not None:
        print(f'[INFO] first {IMGS_PER_CLASS} indices per class: {indices}')
    else:
        print(f'[WARN] It was not possible to find {IMGS_PER_CLASS} indices for all classes')

In [ ]:
import matplotlib.pyplot as plt

def plot_image_grid(images, indices, nrows, ncolumns, file_save):
    '''
    Make a nrows x ncolumns grid of images. 
    '''
    fig, ax = plt.subplots(nr, nc, figsize=(nc * 2, nr * 2))

    for r in range(nrows):
        for c in range(ncolumns):
            pos_c = indices[c]
            pos   = pos_c[r]
            ax[r][c].imshow(images[pos], cmap=None)  # Adjust colormap if needed
            ax[r][c].axis("off")

    plt.tight_layout()
    plt.savefig(file_save)
    plt.show()

In [ ]:
# plot IMGS_PER_CLASS images of each class, one image of each class per row

fname = os.path.join(
    config.experiment.root_dir,
    config.experiment.results_dir,
    config.experiment.experiment_name,
    f"generated_images_{config.experiment.experiment_name}.png"
)

nr, nc = IMGS_PER_CLASS, 4  # Define grid dimensions

plot_image_grid(session.images, indices, nr, nc, fname)

wandb.log({"generated images": wandb.Image(fname)})

## Finishing the Connection to Weights & Bias

In [ ]:
wandb.finish()